<a href="https://colab.research.google.com/github/falloudev190/Crous-T_local/blob/main/Projet_BDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Bonjour le monde !")

Bonjour le monde !


In [ ]:
import sqlite3

def setup_crous_database():
    conn = sqlite3.connect('crous_vcn.db')
    cursor = conn.cursor()

    # Execution du schema valide sur SQLite Online
    sql_script = """
    PRAGMA foreign_keys = ON;

    CREATE TABLE IF NOT EXISTS utilisateurs (
        id_utilisateur INTEGER PRIMARY KEY AUTOINCREMENT,
        nom TEXT NOT NULL,
        prenom TEXT NOT NULL,
        email TEXT UNIQUE NOT NULL,
        telephone TEXT,
        role TEXT NOT NULL CHECK(role IN ('DEMANDEUR', 'AGENT', 'TECHNICIEN', 'SUPER_ADMIN'))
    );

    CREATE TABLE IF NOT EXISTS demandeurs (
        id_utilisateur INTEGER PRIMARY KEY,
        cni TEXT NOT NULL,
        est_etudiant BOOLEAN NOT NULL DEFAULT 0,
        num_etudiant TEXT,
        FOREIGN KEY (id_utilisateur) REFERENCES utilisateurs(id_utilisateur) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS agents (
        id_utilisateur INTEGER PRIMARY KEY,
        service TEXT NOT NULL CHECK(service IN ('COMMISSION', 'FINANCE', 'JURIDIQUE', 'DCUVE')),
        matricule TEXT UNIQUE NOT NULL,
        FOREIGN KEY (id_utilisateur) REFERENCES utilisateurs(id_utilisateur) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS techniciens (
        id_utilisateur INTEGER PRIMARY KEY,
        specialite TEXT NOT NULL,
        disponibilite BOOLEAN DEFAULT 1,
        FOREIGN KEY (id_utilisateur) REFERENCES utilisateurs(id_utilisateur) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS locaux_commerciaux (
        id_local INTEGER PRIMARY KEY AUTOINCREMENT,
        code_local TEXT UNIQUE NOT NULL,
        type_activite TEXT NOT NULL,
        surface_m2 REAL NOT NULL,
        statut TEXT DEFAULT 'Disponible' CHECK(statut IN ('Disponible', 'Occupé', 'En attente', 'Maintenance'))
    );

    CREATE TABLE IF NOT EXISTS demandes_local (
        id_demande INTEGER PRIMARY KEY AUTOINCREMENT,
        code_demande TEXT UNIQUE NOT NULL,
        type_demande TEXT NOT NULL CHECK(type_demande IN ('OBTENTION', 'CONSTRUCTION')),
        date_depot DATETIME DEFAULT CURRENT_TIMESTAMP,
        statut TEXT DEFAULT 'EN_ATTENTE' CHECK(statut IN ('EN_ATTENTE', 'EN_COURS', 'VALIDEE', 'REJETEE')),
        id_demandeur INTEGER NOT NULL,
        FOREIGN KEY (id_demandeur) REFERENCES demandeurs(id_utilisateur)
    );

    CREATE TABLE IF NOT EXISTS contrats (
        id_contrat INTEGER PRIMARY KEY AUTOINCREMENT,
        num_contrat TEXT UNIQUE NOT NULL,
        date_signature DATE,
        montant_loyer_mensuel REAL NOT NULL CHECK(montant_loyer_mensuel >= 0),
        valide_par_directeur BOOLEAN DEFAULT 0,
        statut TEXT DEFAULT 'EN_ATTENTE' CHECK(statut IN ('ACTIF', 'RESILIE', 'EN_ATTENTE')),
        id_demande INTEGER UNIQUE,
        id_local INTEGER NOT NULL,
        FOREIGN KEY (id_demande) REFERENCES demandes_local(id_demande),
        FOREIGN KEY (id_local) REFERENCES locaux_commerciaux(id_local)
    );

    CREATE TABLE IF NOT EXISTS paiements (
        id_paiement INTEGER PRIMARY KEY AUTOINCREMENT,
        montant REAL NOT NULL CHECK(montant > 0),
        date_paiement DATETIME DEFAULT CURRENT_TIMESTAMP,
        mode_paiement TEXT NOT NULL,
        id_contrat INTEGER NOT NULL,
        FOREIGN KEY (id_contrat) REFERENCES contrats(id_contrat)
    );
    """
    cursor.executescript(sql_script)

    # Seeding des utilisateurs
    cursor.execute("SELECT COUNT(*) FROM utilisateurs")
    if cursor.fetchone()[0] == 0:
        cursor.execute("INSERT INTO utilisateurs (nom, prenom, email, telephone, role) VALUES ('Admin', 'Super', 'admin@crous.sn', '770000000', 'SUPER_ADMIN')")

        cursor.execute("INSERT INTO utilisateurs (nom, prenom, email, telephone, role) VALUES ('Faye', 'Awa', 'awa.faye@crous.sn', '771112233', 'AGENT')")
        u_comm = cursor.lastrowid
        cursor.execute("INSERT INTO agents (id_utilisateur, service, matricule) VALUES (?, 'COMMISSION', 'AGT-COMM-01')", (u_comm,))

        cursor.execute("INSERT INTO utilisateurs (nom, prenom, email, telephone, role) VALUES ('Sene', 'Ousmane', 'ousmane.sene@crous.sn', '774445566', 'AGENT')")
        u_fin = cursor.lastrowid
        cursor.execute("INSERT INTO agents (id_utilisateur, service, matricule) VALUES (?, 'FINANCE', 'AGT-FIN-01')", (u_fin,))

        cursor.execute("INSERT INTO utilisateurs (nom, prenom, email, telephone, role) VALUES ('Ndiaye', 'Modou', 'modou.ndiaye@crous.sn', '778889900', 'TECHNICIEN')")
        u_tech = cursor.lastrowid
        cursor.execute("INSERT INTO techniciens (id_utilisateur, specialite, disponibilite) VALUES (?, 'Plomberie & Électricité', 1)", (u_tech,))

    conn.commit()

    # Verification des tables generees
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0] for row in cursor.fetchall() if row[0] not in ('sqlite_sequence', 'demo')]
    conn.close()

    print(f" Base de données créée dans Colab avec les tables : {', '.join(tables)}")

setup_crous_database()
# Une fois cette cellule exécutée dans Colab, le fichier `crous_vcn.db` sera disponible localement dans votre environnement de travail avec l'ensemble des contraintes de clés étrangères et la hiérarchie des rôles.

 Base de données créée dans Colab avec les tables : utilisateurs, demandeurs, agents, techniciens, locaux_commerciaux, demandes_local, contrats, paiements


In [ ]:
# 1. Sauvegarder le code complet dans index.html
with open("index.html", "w", encoding="utf-8") as f:
    f.write("""<!DOCTYPE html>
<html lang="fr">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>Kaay Job THIES — Gestion des espaces commerciaux du campus CROUS</title>

  <!-- Typography -->
  <link rel="preconnect" href="https://fonts.googleapis.com" />
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin />
  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500;600&family=Poppins:wght@400;500;600;700;800;900&display=swap" rel="stylesheet" />

  <style>
    /* CSS DESIGN SYSTEM & TAILWIND COLORS INTEGRATION */
    :root {
      --font-sans: 'Inter', system-ui, -apple-system, sans-serif;
      --font-heading: 'Poppins', system-ui, sans-serif;
      --font-mono: 'JetBrains Mono', monospace;

      --color-primary: #2563EB;
      --color-primary-dark: #1D4ED8;
      --color-primary-light: #DBEAFE;
      --color-success: #22C55E;
      --color-success-light: #D1FAE5;
      --color-warning: #F59E0B;
      --color-warning-light: #FEF3C7;
      --color-danger: #EF4444;
      --color-danger-light: #FEE2E2;

      /* Sidebar Theme from Specs */
      --color-sidebar: #0F172A;
      --color-sidebar-text: #94A3B8;
      --color-sidebar-hover: #1E293B;
      --color-sidebar-active: #1D4ED8;

      --bg-light: #F8FAFC;
      --card-bg: #FFFFFF;
      --text-main: #0F172A;
      --text-muted: #64748B;
      --border-color: #E2E8F0;
    }

    *, *::before, *::after {
      box-sizing: border-box;
      margin: 0;
      padding: 0;
    }

    html {
      scroll-behavior: smooth;
      height: 100%;
    }

    body {
      font-family: var(--font-sans);
      background: var(--bg-light);
      color: var(--text-main);
      -webkit-font-smoothing: antialiased;
      -moz-osx-font-smoothing: grayscale;
      overflow-x: hidden;
      line-height: 1.5;
    }

    a {
      text-decoration: none;
      color: inherit;
    }

    button, input, select, textarea {
      font-family: inherit;
      border: none;
      outline: none;
    }

    button {
      cursor: pointer;
    }

    /* Custom Scrollbar */
    ::-webkit-scrollbar {
      width: 6px;
      height: 6px;
    }
    ::-webkit-scrollbar-track {
      background: transparent;
    }
    ::-webkit-scrollbar-thumb {
      background: #CBD5E1;
      border-radius: 3px;
    }
    ::-webkit-scrollbar-thumb:hover {
      background: #94A3B8;
    }

    /* Layout Utilities */
    .container {
      max-width: 1280px;
      margin: 0 auto;
      padding: 0 2rem;
    }

    .text-gradient {
      background: linear-gradient(135deg, #2563EB 0%, #7C3AED 100%);
      -webkit-background-clip: text;
      -webkit-text-fill-color: transparent;
      background-clip: text;
    }

    .badge {
      display: inline-flex;
      align-items: center;
      gap: 8px;
      padding: 6px 16px;
      border-radius: 999px;
      font-size: 12px;
      font-weight: 600;
      background: var(--color-primary-light);
      color: var(--color-primary-dark);
    }

    .section-title {
      font-family: var(--font-heading);
      font-size: clamp(28px, 4vw, 42px);
      font-weight: 700;
      color: var(--text-main);
      margin-bottom: 16px;
      line-height: 1.2;
    }

    .section-sub {
      font-size: 17px;
      color: var(--text-muted);
      max-width: 540px;
      line-height: 1.7;
    }

    /* Animations */
    @keyframes float {
      0%, 100% { transform: translateY(0); }
      50% { transform: translateY(-10px); }
    }
    @keyframes pulse-dot {
      0%, 100% { opacity: 1; transform: scale(1); }
      50% { opacity: 0.5; transform: scale(0.8); }
    }
    @keyframes fadeUp {
      from { opacity: 0; transform: translateY(24px); }
      to { opacity: 1; transform: translateY(0); }
    }
    @keyframes slideInRight {
      from { transform: translateX(100%); }
      to { transform: translateX(0); }
    }
    @keyframes fadeIn {
      from { opacity: 0; }
      to { opacity: 1; }
    }

    .float-anim { animation: float 6s ease-in-out infinite; }
    .pulse { animation: pulse-dot 2s ease-in-out infinite; }

    .card-hover {
      transition: transform 0.3s cubic-bezier(.34,1.56,.64,1), box-shadow 0.3s ease;
    }
    .card-hover:hover {
      transform: translateY(-5px);
      box-shadow: 0 20px 40px rgba(37,99,235,0.12) !important;
    }

    /* TOP NAVBAR */
    #navbar {
      position: fixed;
      top: 0;
      left: 0;
      right: 0;
      z-index: 100;
      transition: all 0.4s ease;
      background: rgba(255,255,255,0.9);
      backdrop-filter: blur(16px);
      border-bottom: 1px solid rgba(226,232,240,0.8);
    }
    #navbar.scrolled {
      background: rgba(255,255,255,0.95);
      box-shadow: 0 4px 30px rgba(0,0,0,0.06);
    }
    .nav-inner {
      display: flex;
      align-items: center;
      justify-content: space-between;
      height: 76px;
    }
    .nav-logo {
      display: flex;
      align-items: center;
      gap: 12px;
      text-decoration: none;
    }
    .nav-logo-icon {
      width: 40px;
      height: 40px;
      border-radius: 12px;
      background: linear-gradient(135deg, #2563EB, #1D4ED8);
      display: flex;
      align-items: center;
      justify-content: center;
      color: white;
      font-weight: 800;
      box-shadow: 0 4px 12px rgba(37,99,235,0.3);
    }
    .nav-logo-text {
      font-family: var(--font-heading);
      font-size: 22px;
      font-weight: 800;
      color: #0F172A;
      letter-spacing: -0.5px;
    }
    .nav-logo-text span {
      color: var(--color-primary);
    }
    .nav-links {
      display: flex;
      align-items: center;
      gap: 8px;
    }
    .nav-links a {
      padding: 8px 16px;
      border-radius: 10px;
      font-size: 14px;
      font-weight: 500;
      color: #475569;
      transition: all 0.2s;
    }
    .nav-links a:hover {
      background: #EFF6FF;
      color: var(--color-primary);
    }
    .nav-actions {
      display: flex;
      align-items: center;
      gap: 12px;
    }
    .btn-ghost {
      padding: 10px 18px;
      border-radius: 12px;
      font-size: 14px;
      font-weight: 600;
      color: #334155;
      background: transparent;
      transition: background 0.2s;
    }
    .btn-ghost:hover {
      background: #F1F5F9;
    }
    .btn-primary {
      padding: 10px 22px;
      border-radius: 12px;
      font-size: 14px;
      font-weight: 600;
      color: white;
      background: linear-gradient(135deg, #2563EB, #1D4ED8);
      box-shadow: 0 4px 14px rgba(37,99,235,0.35);
      transition: transform 0.2s, box-shadow 0.2s;
      display: inline-flex;
      align-items: center;
      gap: 8px;
    }
    .btn-primary:hover {
      transform: translateY(-2px);
      box-shadow: 0 8px 24px rgba(37,99,235,0.45);
    }
    .hamburger {
      display: none;
      background: none;
      padding: 8px;
      border-radius: 8px;
    }
    .mobile-menu {
      display: none;
      padding: 16px 24px 24px;
      background: rgba(255,255,255,0.98);
      backdrop-filter: blur(20px);
      border-top: 1px solid #F1F5F9;
    }
    .mobile-menu.open {
      display: block;
    }
    .mobile-menu a {
      display: block;
      padding: 12px 16px;
      border-radius: 8px;
      font-size: 15px;
      font-weight: 500;
      color: #334155;
      margin-bottom: 4px;
    }

    /* HERO SECTION */
    #hero {
      min-height: 92vh;
      display: flex;
      align-items: center;
      background: linear-gradient(160deg, #EFF6FF 0%, #F8FAFC 45%, #F0FDF4 100%);
      padding: 120px 0 80px;
      position: relative;
      overflow: hidden;
    }
    #hero::before {
      content: '';
      position: absolute;
      inset: 0;
      background-image: linear-gradient(rgba(37,99,235,0.04) 1px, transparent 1px), linear-gradient(90deg, rgba(37,99,235,0.04) 1px, transparent 1px);
      background-size: 60px 60px;
    }
    .hero-grid {
      display: grid;
      grid-template-columns: 1.1fr 1fr;
      gap: 56px;
      align-items: center;
      position: relative;
      z-index: 1;
    }
    .hero-left {
      display: flex;
      flex-direction: column;
      gap: 28px;
      animation: fadeUp 0.7s ease both;
    }
    .hero-badge {
      display: inline-flex;
      align-items: center;
      gap: 10px;
      padding: 8px 18px;
      border-radius: 999px;
      font-size: 13px;
      font-weight: 600;
      background: #DBEAFE;
      color: #1D4ED8;
      border: 1px solid #BFDBFE;
      width: fit-content;
    }
    .hero-title {
      font-family: var(--font-heading);
      font-size: clamp(38px, 4.8vw, 58px);
      font-weight: 800;
      line-height: 1.12;
      color: #0F172A;
      letter-spacing: -1px;
    }
    .hero-sub {
      font-size: 18px;
      color: #64748B;
      line-height: 1.7;
      max-width: 520px;
    }
    .hero-ctas {
      display: flex;
      gap: 16px;
      flex-wrap: wrap;
      margin-top: 4px;
    }
    .btn-hero-p {
      display: inline-flex;
      align-items: center;
      gap: 10px;
      padding: 16px 30px;
      border-radius: 16px;
      font-size: 15px;
      font-weight: 600;
      color: white;
      background: linear-gradient(135deg, #2563EB, #1D4ED8);
      box-shadow: 0 8px 24px rgba(37,99,235,0.35);
      transition: transform 0.2s, box-shadow 0.2s;
    }
    .btn-hero-p:hover {
      transform: translateY(-2px);
      box-shadow: 0 14px 32px rgba(37,99,235,0.45);
    }
    .btn-hero-s {
      display: inline-flex;
      align-items: center;
      gap: 10px;
      padding: 16px 28px;
      border-radius: 16px;
      font-size: 15px;
      font-weight: 600;
      color: #334155;
      background: white;
      border: 1px solid #E2E8F0;
      box-shadow: 0 2px 8px rgba(0,0,0,0.04);
      transition: border-color 0.2s, color 0.2s;
    }
    .btn-hero-s:hover {
      border-color: #2563EB;
      color: #2563EB;
    }
    .hero-trust {
      display: flex;
      align-items: center;
      gap: 16px;
      margin-top: 8px;
    }
    .avatars {
      display: flex;
    }
    .av {
      width: 38px;
      height: 38px;
      border-radius: 50%;
      border: 2.5px solid white;
      display: flex;
      align-items: center;
      justify-content: center;
      color: white;
      font-size: 12px;
      font-weight: 700;
      margin-left: -10px;
      box-shadow: 0 2px 6px rgba(0,0,0,0.1);
    }
    .avatars .av:first-child { margin-left: 0; }
    .trust-text { font-size: 13px; color: #64748B; }
    .trust-text strong { color: #1E293B; font-weight: 700; }
    .stars { color: #F59E0B; font-size: 14px; margin-bottom: 2px; }

    /* CAMPUS GRAPHIC & SCENE */
    .campus-wrap {
      border-radius: 28px;
      overflow: hidden;
      box-shadow: 0 40px 80px rgba(37,99,235,0.15), 0 0 0 1px rgba(37,99,235,0.08);
      position: relative;
      background: white;
    }
    .campus-scene {
      width: 100%;
      position: relative;
      background: linear-gradient(180deg, #BFDBFE 0%, #EFF6FF 40%, #D1FAE5 100%);
    }
    .fc {
      position: absolute;
      background: white;
      border-radius: 16px;
      padding: 12px 16px;
      border: 1px solid #E2E8F0;
      box-shadow: 0 12px 32px rgba(0,0,0,0.08);
      display: flex;
      align-items: center;
      gap: 12px;
      z-index: 5;
    }
    .fc-icon {
      width: 36px;
      height: 36px;
      border-radius: 10px;
      display: flex;
      align-items: center;
      justify-content: center;
      flex-shrink: 0;
    }
    .fc-title { font-size: 13px; font-weight: 600; color: #1F2937; }
    .fc-sub { font-size: 11px; color: #94A3B8; }
    .fc1 { top: 20px; left: 20px; animation: float 6s ease-in-out infinite; }
    .fc2 { top: 20px; right: 20px; animation: float 6s ease-in-out infinite; animation-delay: -2s; flex-direction: column; align-items: flex-start; gap: 6px; }
    .fc3 { bottom: 90px; left: 20px; animation: float 6s ease-in-out infinite; animation-delay: -1s; flex-direction: column; align-items: flex-start; gap: 8px; min-width: 160px; }
    .fc4 { bottom: 90px; right: 20px; animation: float 6s ease-in-out infinite; animation-delay: -3s; }
    .mini-chart { display: flex; align-items: flex-end; gap: 4px; height: 42px; width: 100%; }
    .mb { flex: 1; border-radius: 3px; }

    /* STATS SECTION */
    #stats { padding: 80px 0; background: white; border-bottom: 1px solid #F1F5F9; }
    .stats-grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 24px; }
    .stat-card {
      background: white;
      border-radius: 24px;
      padding: 32px 28px;
      border: 1px solid #F1F5F9;
      box-shadow: 0 4px 20px rgba(0,0,0,0.03);
    }
    .stat-icon {
      width: 52px;
      height: 52px;
      border-radius: 16px;
      display: flex;
      align-items: center;
      justify-content: center;
      margin-bottom: 20px;
    }
    .stat-value {
      font-family: var(--font-heading);
      font-size: 42px;
      font-weight: 800;
      line-height: 1;
      margin-bottom: 8px;
    }
    .stat-label { font-size: 14px; font-weight: 500; color: #64748B; }

    /* HOW IT WORKS */
    #how { padding: 96px 0; background: #F8FAFC; }
    .timeline {
      display: grid;
      grid-template-columns: repeat(8, 1fr);
      gap: 16px;
      position: relative;
    }
    .timeline::before {
      content: '';
      position: absolute;
      top: 24px;
      left: calc(100% / 16);
      right: calc(100% / 16);
      height: 2px;
      background: linear-gradient(90deg, #2563EB, #7C3AED, #22C55E);
      opacity: 0.3;
    }
    .step {
      display: flex;
      flex-direction: column;
      align-items: center;
      text-align: center;
    }
    .step-icon {
      width: 52px;
      height: 52px;
      border-radius: 18px;
      display: flex;
      align-items: center;
      justify-content: center;
      margin-bottom: 16px;
      position: relative;
      z-index: 1;
      box-shadow: 0 4px 14px rgba(0,0,0,0.06);
      transition: transform 0.25s;
    }
    .step-icon:hover { transform: scale(1.15); }
    .step-num {
      position: absolute;
      top: -6px;
      right: -6px;
      width: 22px;
      height: 22px;
      border-radius: 50%;
      display: flex;
      align-items: center;
      justify-content: center;
      color: white;
      font-size: 10px;
      font-weight: 800;
      border: 2px solid white;
    }
    .step-t { font-size: 14px; font-weight: 700; color: #0F172A; font-family: var(--font-heading); margin-bottom: 4px; }
    .step-d { font-size: 12px; color: #64748B; line-height: 1.5; }

    /* FEATURES */
    #features { padding: 96px 0; background: white; }
    .features-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 28px; }
    .feat-card {
      border-radius: 24px;
      padding: 36px 32px;
      border: 1px solid rgba(226,232,240,0.8);
      cursor: pointer;
    }
    .feat-icon {
      width: 52px;
      height: 52px;
      border-radius: 16px;
      background: white;
      display: flex;
      align-items: center;
      justify-content: center;
      margin-bottom: 24px;
      box-shadow: 0 4px 14px rgba(0,0,0,0.06);
    }
    .feat-title { font-family: var(--font-heading); font-size: 18px; font-weight: 700; color: #0F172A; margin-bottom: 10px; }
    .feat-desc { font-size: 14px; color: #64748B; line-height: 1.6; }

    /* MAP SECTION */
    #map { padding: 96px 0; background: #F8FAFC; }
    .map-layout { display: grid; grid-template-columns: 2fr 3fr; gap: 64px; align-items: start; }
    .map-legend { display: flex; flex-direction: column; gap: 12px; margin: 28px 0; }
    .leg-item { display: flex; align-items: center; gap: 12px; font-size: 14px; font-weight: 600; color: #334155; }
    .leg-dot { width: 14px; height: 14px; border-radius: 50%; flex-shrink: 0; }
    .map-detail {
      background: white;
      border-radius: 20px;
      padding: 24px;
      border: 1px solid #E2E8F0;
      box-shadow: 0 8px 30px rgba(0,0,0,0.06);
      margin-bottom: 24px;
      display: none;
    }
    .map-detail.vis { display: block; animation: fadeIn 0.3s ease; }
    .map-detail-head { display: flex; align-items: center; justify-content: space-between; margin-bottom: 10px; }
    .map-dname { font-weight: 700; color: #0F172A; font-size: 16px; font-family: var(--font-heading); }
    .map-tag { padding: 4px 12px; border-radius: 999px; font-size: 12px; font-weight: 700; }
    .map-dcat { font-size: 14px; color: #64748B; margin-bottom: 18px; }
    .btn-req {
      width: 100%;
      padding: 12px;
      border-radius: 12px;
      font-size: 14px;
      font-weight: 600;
      color: white;
      background: linear-gradient(135deg, #2563EB, #1D4ED8);
    }
    .campus-svg-wrap {
      width: 100%;
      height: 460px;
      border-radius: 28px;
      border: 1px solid #E2E8F0;
      box-shadow: 0 20px 60px rgba(0,0,0,0.08);
      position: relative;
      overflow: hidden;
      background: linear-gradient(135deg, #E0F2FE 0%, #D1FAE5 50%, #FEF3C7 100%);
    }
    .map-pin {
      position: absolute;
      transform: translate(-50%, -50%);
      width: 26px;
      height: 26px;
      border-radius: 50%;
      border: 3px solid white;
      box-shadow: 0 4px 12px rgba(0,0,0,0.25);
      cursor: pointer;
      transition: transform 0.25s cubic-bezier(.34,1.56,.64,1);
      display: flex;
      align-items: center;
      justify-content: center;
    }
    .map-pin:hover, .map-pin.act { transform: translate(-50%, -50%) scale(1.4); z-index: 10; }
    .pin-in { width: 8px; height: 8px; border-radius: 50%; background: white; display: none; }
    .map-pin.act .pin-in { display: block; }
    .btn-exp-map {
      display: inline-flex;
      align-items: center;
      gap: 10px;
      padding: 14px 26px;
      border-radius: 16px;
      font-size: 14px;
      font-weight: 600;
      color: white;
      background: linear-gradient(135deg, #2563EB, #1D4ED8);
      box-shadow: 0 6px 20px rgba(37,99,235,0.3);
    }

    /* CATEGORIES */
    #categories { padding: 96px 0; background: white; }
    .cats-scroll {
      display: flex;
      gap: 20px;
      overflow-x: auto;
      padding-bottom: 16px;
      margin: 0 -2rem;
      padding-left: 2rem;
      padding-right: 2rem;
      scrollbar-width: none;
    }
    .cats-scroll::-webkit-scrollbar { display: none; }
    .cat-card {
      flex-shrink: 0;
      width: 230px;
      border-radius: 24px;
      padding: 28px;
      cursor: pointer;
      border: 1px solid rgba(226,232,240,0.8);
    }
    .cat-emoji { font-size: 40px; margin-bottom: 16px; }
    .cat-name { font-family: var(--font-heading); font-size: 18px; font-weight: 700; color: #0F172A; margin-bottom: 6px; }
    .cat-desc { font-size: 13px; color: #64748B; margin-bottom: 18px; line-height: 1.5; }
    .cat-foot { display: flex; align-items: center; justify-content: space-between; }
    .cat-count { font-size: 13px; font-weight: 700; }
    .btn-expl {
      display: flex;
      align-items: center;
      gap: 6px;
      font-size: 12px;
      font-weight: 600;
      background: white;
      border-radius: 8px;
      padding: 6px 12px;
      box-shadow: 0 2px 6px rgba(0,0,0,0.06);
    }

    /* WHY SECTION */
    #why { padding: 96px 0; background: #F8FAFC; }
    .why-grid { display: grid; grid-template-columns: 1.1fr 1fr; gap: 64px; align-items: center; }
    .reasons { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-top: 36px; }
    .reason { background: white; border-radius: 20px; padding: 22px; border: 1px solid #F1F5F9; box-shadow: 0 2px 12px rgba(0,0,0,0.03); }
    .reason-icon { width: 42px; height: 42px; border-radius: 14px; display: flex; align-items: center; justify-content: center; margin-bottom: 16px; }
    .reason-t { font-family: var(--font-heading); font-size: 15px; font-weight: 700; color: #0F172A; margin-bottom: 6px; }
    .reason-d { font-size: 12px; color: #64748B; line-height: 1.6; }

    .dash-wrap { background: linear-gradient(135deg, #EFF6FF, #F5F3FF); border-radius: 28px; padding: 32px; border: 1px solid #DBEAFE; }
    .dash-card { background: white; border-radius: 20px; padding: 24px; box-shadow: 0 4px 20px rgba(0,0,0,0.06); margin-bottom: 16px; }
    .dash-head { display: flex; justify-content: space-between; align-items: center; margin-bottom: 16px; }
    .dash-head-t { font-size: 15px; font-weight: 700; color: #1E293B; }
    .online { padding: 4px 10px; border-radius: 999px; font-size: 12px; font-weight: 600; background: #D1FAE5; color: #22C55E; }
    .dash-metrics { display: grid; grid-template-columns: repeat(3,1fr); gap: 12px; margin-bottom: 16px; }
    .dm { border-radius: 14px; padding: 14px; text-align: center; }
    .dm-val { font-family: var(--font-heading); font-size: 24px; font-weight: 800; }
    .dm-lbl { font-size: 12px; color: #64748B; margin-top: 2px; }
    .dash-row { display: flex; align-items: center; justify-content: space-between; padding: 10px 0; border-bottom: 1px solid #F8FAFC; }
    .dash-person { display: flex; align-items: center; gap: 10px; }
    .dav { width: 32px; height: 32px; border-radius: 50%; display: flex; align-items: center; justify-content: center; color: white; font-size: 12px; font-weight: 700; }
    .dname { font-size: 13px; font-weight: 500; color: #334155; }
    .stag { padding: 3px 10px; border-radius: 999px; font-size: 11px; font-weight: 700; }
    .dash-bottom { display: grid; grid-template-columns: 1fr 1fr; gap: 12px; }
    .dmini { background: white; border-radius: 18px; padding: 18px; box-shadow: 0 4px 20px rgba(0,0,0,0.06); }
    .dmini-lbl { font-size: 12px; color: #64748B; margin-bottom: 4px; }
    .dmini-val { font-family: var(--font-heading); font-size: 22px; font-weight: 800; }
    .prog { height: 6px; border-radius: 3px; background: #F1F5F9; margin-top: 10px; }
    .prog-fill { height: 100%; border-radius: 3px; }
    .trend { display: flex; align-items: center; gap: 4px; font-size: 12px; margin-top: 6px; font-weight: 600; }

    /* TESTIMONIALS */
    #testimonials { padding: 96px 0; background: white; }
    .testi-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 28px; }
    .testi { background: white; border-radius: 28px; padding: 36px; border: 1px solid #F1F5F9; box-shadow: 0 4px 24px rgba(0,0,0,0.04); }
    .testi-stars { color: #F59E0B; font-size: 16px; margin-bottom: 20px; }
    .testi-text { font-size: 15px; color: #334155; line-height: 1.7; font-style: italic; margin-bottom: 28px; }
    .testi-person { display: flex; align-items: center; gap: 14px; }
    .tav { width: 46px; height: 46px; border-radius: 50%; display: flex; align-items: center; justify-content: center; color: white; font-size: 15px; font-weight: 700; }
    .tname { font-family: var(--font-heading); font-size: 15px; font-weight: 700; color: #0F172A; }
    .trole { font-size: 12px; color: #64748B; }
    .tuni { font-size: 11px; color: #94A3B8; }

    /* FAQ */
    #faq { padding: 96px 0; background: #F8FAFC; }
    .faq-list { max-width: 760px; margin: 0 auto; display: flex; flex-direction: column; gap: 14px; }
    .faq-item { background: white; border-radius: 20px; overflow: hidden; border: 1px solid #E2E8F0; box-shadow: 0 2px 8px rgba(0,0,0,0.03); transition: box-shadow 0.3s; }
    .faq-item.open { box-shadow: 0 8px 24px rgba(37,99,235,0.08); border-color: #BFDBFE; }
    .faq-q { width: 100%; display: flex; align-items: center; justify-content: space-between; padding: 22px 26px; background: none; text-align: left; font-family: var(--font-heading); font-size: 16px; font-weight: 600; color: #0F172A; cursor: pointer; }
    .faq-tog { width: 32px; height: 32px; flex-shrink: 0; border-radius: 50%; background: #F1F5F9; display: flex; align-items: center; justify-content: center; transition: background 0.3s, transform 0.3s; }
    .faq-item.open .faq-tog { background: #2563EB; color: white; transform: rotate(180deg); }
    .faq-item.open .faq-tog svg { stroke: white; }
    .faq-ans { max-height: 0; overflow: hidden; transition: max-height 0.35s ease; }
    .faq-item.open .faq-ans { max-height: 220px; }
    .faq-ans-in { padding: 0 26px 24px; font-size: 14px; color: #64748B; line-height: 1.7; }

    /* CTA */
    #cta { padding: 80px 0; }
    .cta-box {
      border-radius: 32px;
      padding: 80px 40px;
      text-align: center;
      background: linear-gradient(135deg, #1E40AF 0%, #2563EB 50%, #3B82F6 100%);
      position: relative;
      overflow: hidden;
      box-shadow: 0 20px 50px rgba(37,99,235,0.3);
    }
    .cg1 { position: absolute; top: -60px; left: -60px; width: 280px; height: 280px; border-radius: 50%; background: rgba(255,255,255,0.08); }
    .cg2 { position: absolute; bottom: -80px; right: -80px; width: 380px; height: 380px; border-radius: 50%; background: rgba(255,255,255,0.08); }
    .cta-in { position: relative; z-index: 1; }
    .cta-tag { display: inline-flex; align-items: center; gap: 6px; padding: 6px 18px; border-radius: 999px; font-size: 13px; font-weight: 600; background: rgba(255,255,255,0.15); color: white; border: 1px solid rgba(255,255,255,0.25); margin-bottom: 24px; }
    .cta-title { font-family: var(--font-heading); font-size: clamp(30px, 4vw, 46px); font-weight: 800; color: white; line-height: 1.2; margin-bottom: 20px; }
    .cta-sub { font-size: 17px; color: rgba(255,255,255,0.85); max-width: 600px; margin: 0 auto 40px; line-height: 1.6; }
    .cta-btns { display: flex; gap: 16px; justify-content: center; flex-wrap: wrap; }
    .btn-cta-w { padding: 16px 36px; border-radius: 16px; font-size: 15px; font-weight: 700; color: #2563EB; background: white; box-shadow: 0 8px 24px rgba(0,0,0,0.15); transition: transform 0.2s, box-shadow 0.2s; }
    .btn-cta-w:hover { transform: translateY(-2px); box-shadow: 0 14px 32px rgba(0,0,0,0.22); }
    .btn-cta-g { padding: 16px 32px; border-radius: 16px; font-size: 15px; font-weight: 600; color: white; background: rgba(255,255,255,0.12); border: 1px solid rgba(255,255,255,0.35); transition: background 0.2s; }
    .btn-cta-g:hover { background: rgba(255,255,255,0.22); }

    /* FOOTER */
    #footer { background: white; border-top: 1px solid #F1F5F9; padding: 64px 0 32px; }
    .footer-grid { display: grid; grid-template-columns: 2fr 1fr 1fr 1fr 1fr; gap: 40px; margin-bottom: 48px; }
    .footer-desc { font-size: 14px; color: #64748B; line-height: 1.6; max-width: 250px; margin: 20px 0 24px; }
    .footer-socials { display: flex; gap: 10px; margin-bottom: 24px; }
    .soc-btn { width: 38px; height: 38px; border-radius: 10px; background: none; border: 1px solid #E2E8F0; display: flex; align-items: center; justify-content: center; transition: background 0.2s; }
    .soc-btn:hover { background: #EFF6FF; border-color: #BFDBFE; }
    .fcol h4 { font-family: var(--font-heading); font-size: 14px; font-weight: 700; color: #0F172A; margin-bottom: 20px; }
    .fcol ul { list-style: none; display: flex; flex-direction: column; gap: 12px; }
    .fcol ul li a { font-size: 14px; color: #64748B; transition: color 0.2s; }
    .fcol ul li a:hover { color: var(--color-primary); }
    .foot-contacts { display: flex; flex-wrap: wrap; gap: 24px; padding: 24px 0; border-top: 1px solid #F1F5F9; border-bottom: 1px solid #F1F5F9; margin-bottom: 28px; }
    .fcontact { display: flex; align-items: center; gap: 10px; }
    .fci { width: 32px; height: 32px; border-radius: 10px; background: #DBEAFE; display: flex; align-items: center; justify-content: center; }
    .fct { font-size: 14px; color: #475569; font-weight: 500; }
    .foot-bottom { display: flex; align-items: center; justify-content: space-between; flex-wrap: wrap; gap: 12px; }
    .foot-copy { font-size: 13px; color: #94A3B8; }
    .status-pill { display: flex; align-items: center; gap: 8px; font-size: 12px; font-weight: 600; color: #22C55E; background: #F0FDF4; padding: 6px 14px; border-radius: 999px; border: 1px solid #DCFCE7; }
    .sdot { width: 8px; height: 8px; border-radius: 50%; background: #22C55E; }

    /* DASHBOARD PORTAL & DRAWER OVERLAY */
    .drawer-overlay {
      position: fixed;
      inset: 0;
      z-index: 999;
      background: rgba(15, 23, 42, 0.6);
      backdrop-filter: blur(4px);
      -webkit-backdrop-filter: blur(4px);
      display: none;
      opacity: 0;
      transition: opacity 0.3s ease;
    }
    .drawer-overlay.active {
      display: flex;
      opacity: 1;
    }

    .dashboard-modal {
      width: 100%;
      height: 100%;
      max-width: 1360px;
      max-height: 90vh;
      margin: auto;
      background: #F8FAFC;
      border-radius: 24px;
      box-shadow: 0 25px 50px -12px rgba(0, 0, 0, 0.25);
      display: flex;
      overflow: hidden;
      position: relative;
      animation: slideInRight 0.3s cubic-bezier(0.16, 1, 0.3, 1);
    }

    /* DARK SIDEBAR ACCORDING TO SPECIFICATIONS */
    .app-sidebar {
      width: 280px;
      background-color: var(--color-sidebar); /* #0F172A */
      color: var(--color-sidebar-text);       /* #94A3B8 */
      display: flex;
      flex-direction: column;
      justify-content: space-between;
      padding: 24px 16px;
      flex-shrink: 0;
    }
    .sidebar-header {
      display: flex;
      align-items: center;
      gap: 12px;
      padding: 0 12px 24px;
      border-bottom: 1px solid #1E293B;
    }
    .sidebar-logo-icon {
      width: 36px;
      height: 36px;
      border-radius: 10px;
      background: var(--color-primary);
      color: white;
      display: flex;
      align-items: center;
      justify-content: center;
      font-weight: 800;
    }
    .sidebar-logo-title {
      font-family: var(--font-heading);
      font-weight: 700;
      color: white;
      font-size: 18px;
    }
    .sidebar-logo-title span { color: #60A5FA; }

    .sidebar-menu {
      list-style: none;
      margin-top: 20px;
      display: flex;
      flex-direction: column;
      gap: 4px;
    }
    .sidebar-item {
      display: flex;
      align-items: center;
      gap: 12px;
      padding: 12px 16px;
      border-radius: 12px;
      font-size: 14px;
      font-weight: 500;
      color: var(--color-sidebar-text);
      cursor: pointer;
      transition: all 0.2s ease;
    }
    .sidebar-item:hover {
      background-color: var(--color-sidebar-hover); /* #1E293B */
      color: #F8FAFC;
    }
    .sidebar-item.active {
      background-color: var(--color-sidebar-active); /* #1D4ED8 */
      color: #FFFFFF;
      font-weight: 600;
      box-shadow: 0 4px 14px rgba(29, 78, 216, 0.4);
    }
    .sidebar-item svg {
      width: 20px;
      height: 20px;
      stroke: currentColor;
    }

    .user-profile-card {
      background: var(--color-sidebar-hover);
      border-radius: 16px;
      padding: 14px;
      display: flex;
      align-items: center;
      gap: 12px;
      margin-top: 20px;
    }
    .u-avatar {
      width: 38px;
      height: 38px;
      border-radius: 50%;
      background: #2563EB;
      color: white;
      display: flex;
      align-items: center;
      justify-content: center;
      font-weight: 700;
      font-size: 14px;
    }
    .u-info {
      flex: 1;
      overflow: hidden;
    }
    .u-name { font-size: 13px; font-weight: 600; color: white; white-space: nowrap; text-overflow: ellipsis; overflow: hidden; }
    .u-role { font-size: 11px; color: #94A3B8; }

    /* DASHBOARD MAIN CONTAINER */
    .app-main {
      flex: 1;
      display: flex;
      flex-direction: column;
      overflow-y: auto;
      background: #F8FAFC;
    }

    .app-topbar {
      display: flex;
      align-items: center;
      justify-content: space-between;
      padding: 20px 32px;
      background: white;
      border-bottom: 1px solid #E2E8F0;
    }
    .topbar-title {
      font-family: var(--font-heading);
      font-size: 20px;
      font-weight: 700;
      color: #0F172A;
    }
    .topbar-actions {
      display: flex;
      align-items: center;
      gap: 16px;
    }
    .btn-close-drawer {
      width: 36px;
      height: 36px;
      border-radius: 50%;
      background: #F1F5F9;
      display: flex;
      align-items: center;
      justify-content: center;
      font-size: 18px;
      color: #475569;
      transition: background 0.2s;
    }
    .btn-close-drawer:hover { background: #E2E8F0; color: #0F172A; }

    .app-content {
      padding: 32px;
      flex: 1;
    }

    .tab-content {
      display: none;
    }
    .tab-content.active {
      display: block;
      animation: fadeIn 0.3s ease;
    }

    /* FORM STYLES */
    .form-group {
      margin-bottom: 20px;
    }
    .form-label {
      display: block;
      font-size: 13px;
      font-weight: 600;
      color: #334155;
      margin-bottom: 8px;
    }
    .form-input, .form-select, .form-textarea {
      width: 100%;
      padding: 12px 16px;
      border-radius: 12px;
      border: 1px solid #CBD5E1;
      background: white;
      font-size: 14px;
      color: #0F172A;
      transition: border-color 0.2s, box-shadow 0.2s;
    }
    .form-input:focus, .form-select:focus, .form-textarea:focus {
      border-color: #2563EB;
      box-shadow: 0 0 0 3px rgba(37, 99, 235, 0.15);
    }

    /* DATA TABLE */
    .table-container {
      background: white;
      border-radius: 16px;
      border: 1px solid #E2E8F0;
      overflow: hidden;
      box-shadow: 0 4px 14px rgba(0,0,0,0.03);
    }
    table.data-table {
      width: 100%;
      border-collapse: collapse;
      text-align: left;
      font-size: 14px;
    }
    table.data-table th {
      background: #F8FAFC;
      padding: 14px 20px;
      font-size: 12px;
      font-weight: 700;
      color: #475569;
      text-transform: uppercase;
      letter-spacing: 0.5px;
      border-bottom: 1px solid #E2E8F0;
    }
    table.data-table td {
      padding: 16px 20px;
      border-bottom: 1px solid #F1F5F9;
      color: #1E293B;
    }
    table.data-table tr:last-child td { border-bottom: none; }

    .status-badge {
      display: inline-flex;
      align-items: center;
      gap: 6px;
      padding: 4px 10px;
      border-radius: 999px;
      font-size: 12px;
      font-weight: 600;
    }
    .status-badge.approved { background: var(--color-success-light); color: #166534; }
    .status-badge.pending { background: var(--color-warning-light); color: #92400E; }
    .status-badge.rejected { background: var(--color-danger-light); color: #991B1B; }

    /* TOAST NOTIFICATION */
    .toast-container {
      position: fixed;
      bottom: 24px;
      right: 24px;
      z-index: 1000;
      display: flex;
      flex-direction: column;
      gap: 10px;
    }
    .toast {
      background: #0F172A;
      color: white;
      padding: 14px 20px;
      border-radius: 12px;
      box-shadow: 0 10px 25px rgba(0,0,0,0.2);
      display: flex;
      align-items: center;
      gap: 12px;
      font-size: 14px;
      font-weight: 500;
      animation: fadeUp 0.3s ease;
    }

    /* RESPONSIVE DESIGN */
    @media (max-width: 1024px) {
      .hero-grid { grid-template-columns: 1fr; gap: 40px; }
      .hero-right { display: none; }
      .stats-grid { grid-template-columns: repeat(2, 1fr); }
      .features-grid { grid-template-columns: repeat(2, 1fr); }
      .timeline { grid-template-columns: repeat(4, 1fr); }
      .timeline::before { display: none; }
      .map-layout { grid-template-columns: 1fr; }
      .why-grid { grid-template-columns: 1fr; }
      .testi-grid { grid-template-columns: repeat(2, 1fr); }
      .footer-grid { grid-template-columns: 1fr 1fr; }
      .nav-links, .nav-actions { display: none; }
      .hamburger { display: flex; }
      .dashboard-modal { flex-direction: column; max-height: 100vh; border-radius: 0; }
      .app-sidebar { width: 100%; height: auto; }
    }

    @media (max-width: 640px) {
      .features-grid { grid-template-columns: 1fr; }
      .testi-grid { grid-template-columns: 1fr; }
      .timeline { grid-template-columns: repeat(2, 1fr); }
      .reasons { grid-template-columns: 1fr; }
      .footer-grid { grid-template-columns: 1fr; }
      .cta-btns { flex-direction: column; align-items: center; }
    }
  </style>
</head>
<body>

<!-- TOP NAVIGATION BAR -->
<header>
  <nav id="navbar">
    <div class="container">
      <div class="nav-inner">
        <a href="#" class="nav-logo">
          <div class="nav-logo-icon">
            <svg width="20" height="20" viewBox="0 0 24 24" fill="none">
              <rect x="3" y="3" width="7" height="7" rx="2" fill="white" opacity="0.9"/>
              <rect x="14" y="3" width="7" height="7" rx="2" fill="white" opacity="0.6"/>
              <rect x="3" y="14" width="7" height="7" rx="2" fill="white" opacity="0.6"/>
              <rect x="14" y="14" width="7" height="7" rx="2" fill="white" opacity="0.9"/>
            </svg>
          </div>
          <span class="nav-logo-text">Kaay<span>Job</span></span>
        </a>

        <div class="nav-links">
          <a href="#hero">Accueil</a>
          <a href="#features">Services</a>
          <a href="#map">Campus</a>
          <a href="#categories">Commerces</a>
          <a href="#faq">FAQ</a>
          <a href="#footer">Contact</a>
        </div>

        <div class="nav-actions">
          <button class="btn-ghost" onclick="openDashboardPortal()">Connexion</button>
          <button class="btn-primary" onclick="openDashboardPortal()">
            <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.2" stroke-linecap="round"><path d="M15 3h4a2 2 0 0 1 2 2v14a2 2 0 0 1-2 2h-4"/><polyline points="10 17 15 12 10 7"/><line x1="15" y1="12" x2="3" y2="12"/></svg>
            Accéder au Portail
          </button>
        </div>

        <button class="hamburger" id="hamburger" aria-label="Menu Mobile">
          <svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="#334155" stroke-width="2" stroke-linecap="round">
            <line x1="3" y1="12" x2="21" y2="12"/><line x1="3" y1="6" x2="21" y2="6"/><line x1="3" y1="18" x2="21" y2="18"/>
          </svg>
        </button>
      </div>
    </div>

    <div class="mobile-menu" id="mobile-menu">
      <a href="#hero">Accueil</a>
      <a href="#features">Services</a>
      <a href="#map">Campus</a>
      <a href="#categories">Commerces</a>
      <a href="#faq">FAQ</a>
      <a href="#footer">Contact</a>
      <div style="display:flex; gap:12px; margin-top:16px;">
        <button class="btn-ghost" style="flex:1; border:1px solid #E2E8F0;" onclick="openDashboardPortal()">Connexion</button>
        <button class="btn-primary" style="flex:1;" onclick="openDashboardPortal()">Espace Gestion</button>
      </div>
    </div>
  </nav>
</header>

<!-- HERO SECTION -->
<section id="hero">
  <div class="container">
    <div class="hero-grid">
      <div class="hero-left">
        <div class="hero-badge">
          <span class="pulse" style="width:8px;height:8px;border-radius:50%;background:#2563EB;display:inline-block;flex-shrink:0;"></span>
          Plateforme Officielle du CROUS de Thiès
        </div>
        <h1 class="hero-title">Digitalisez la gestion des <span class="text-gradient">espaces commerciaux</span> du campus.</h1>
        <p class="hero-sub">Une plateforme intelligente permettant de déposer une demande de local commercial, suivre son dossier en temps réel, signer son contrat et payer en ligne en toute sécurité.</p>

        <div class="hero-ctas">
          <button class="btn-hero-p" onclick="openDashboardPortal('demandes')">
            Faire une demande
            <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="2" stroke-linecap="round"><line x1="5" y1="12" x2="19" y2="12"/><polyline points="12 5 19 12 12 19"/></svg>
          </button>
          <a href="#map" class="btn-hero-s">
            <svg width="16" height="16" viewBox="0 0 24 24" fill="currentColor"><polygon points="5 3 19 12 5 21 5 3"/></svg>
            Carte interactive
          </a>
        </div>

        <div class="hero-trust">
          <div class="avatars">
            <div class="av" style="background:#2563EB;">A</div>
            <div class="av" style="background:#7C3AED;">F</div>
            <div class="av" style="background:#0F766E;">M</div>
            <div class="av" style="background:#B45309;">K</div>
            <div class="av" style="background:#DC2626;">S</div>
          </div>
          <div>
            <div class="stars">★★★★★</div>
            <div class="trust-text">Approuvé par la Direction du <strong>CROUS THIES</strong></div>
          </div>
        </div>
      </div>

      <div class="hero-right">
        <div class="campus-wrap">
          <div class="campus-scene">
            <svg viewBox="0 0 560 520" xmlns="http://www.w3.org/2000/svg" preserveAspectRatio="xMidYMid slice" style="display:block;width:100%;height:auto;">
              <defs>
                <linearGradient id="gsky" x1="0" y1="0" x2="0" y2="1"><stop offset="0%" stop-color="#BFDBFE"/><stop offset="100%" stop-color="#EFF6FF"/></linearGradient>
                <linearGradient id="gtower" x1="0" y1="0" x2="0" y2="1"><stop offset="0%" stop-color="#1E40AF"/><stop offset="100%" stop-color="#2563EB"/></linearGradient>
                <linearGradient id="gmain" x1="0" y1="0" x2="0" y2="1"><stop offset="0%" stop-color="#DBEAFE"/><stop offset="100%" stop-color="#BFDBFE"/></linearGradient>
              </defs>
              <rect width="560" height="520" fill="url(#gsky)"/>
              <circle cx="460" cy="52" r="34" fill="#FCD34D" opacity="0.9"/>
              <ellipse cx="90" cy="68" rx="38" ry="22" fill="white" opacity="0.85"/>
              <ellipse cx="128" cy="62" rx="48" ry="26" fill="white" opacity="0.9"/>
              <rect x="0" y="400" width="560" height="120" fill="#86EFAC"/>
              <rect x="0" y="408" width="560" height="18" fill="#94A3B8" opacity="0.45"/>
              <line x1="0" y1="416" x2="560" y2="416" stroke="white" stroke-width="2" stroke-dasharray="30 20"/>
              <!-- Clock Tower & Campus Buildings -->
              <rect x="248" y="188" width="64" height="120" fill="url(#gtower)" rx="4"/>
              <circle cx="280" cy="176" r="14" fill="#1E40AF" stroke="#60A5FA" stroke-width="2"/>
              <rect x="200" y="280" width="160" height="130" fill="url(#gmain)" rx="6" stroke="#93C5FD" stroke-width="1.5"/>
              <!-- Kiosks -->
              <rect x="30" y="314" width="110" height="96" fill="#FEF3C7" rx="6" stroke="#FCD34D"/>
              <text x="85" y="332" text-anchor="middle" font-family="Poppins,sans-serif" font-size="10" font-weight="700" fill="#92400E">PAPETERIE</text>
              <rect x="420" y="302" width="120" height="110" fill="#D1FAE5" rx="6" stroke="#6EE7B7"/>
              <text x="480" y="320" text-anchor="middle" font-family="Poppins,sans-serif" font-size="9" font-weight="700" fill="#065F46">ALIMENTATION</text>
            </svg>

            <!-- Floating Cards -->
            <div class="fc fc1">
              <div class="fc-icon" style="background:#D1FAE5;">
                <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="#22C55E" stroke-width="2.5" stroke-linecap="round"><polyline points="20 6 9 17 4 12"/></svg>
              </div>
              <div><div class="fc-title">Paiement réussi</div><div class="fc-sub">85 000 FCFA · Via Wave</div></div>
            </div>
            <div class="fc fc2">
              <div style="display:flex;align-items:center;gap:6px;">
                <span class="pulse" style="width:8px;height:8px;border-radius:50%;background:#2563EB;display:inline-block;"></span>
                <span class="fc-title" style="color:#2563EB;">Nouvelle demande</span>
              </div>
              <div class="fc-title">Local C-204 · Papeterie</div>
              <div class="fc-sub">Moussa Diallo</div>
            </div>
            <div class="fc fc3">
              <div style="display:flex;justify-content:space-between;width:100%;align-items:center;">
                <span class="fc-title">Occupation Campus</span>
                <span style="font-size:11px;color:#22C55E;font-weight:700;">87%</span>
              </div>
              <div class="mini-chart">
                <div class="mb" style="height:40%;background:#DBEAFE;"></div>
                <div class="mb" style="height:65%;background:#DBEAFE;"></div>
                <div class="mb" style="height:80%;background:#DBEAFE;"></div>
                <div class="mb" style="height:90%;background:#2563EB;"></div>
                <div class="mb" style="height:70%;background:#DBEAFE;"></div>
              </div>
            </div>
            <div class="fc fc4">
              <div class="fc-icon" style="background:#EDE9FE;">
                <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="#7C3AED" stroke-width="2" stroke-linecap="round"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/><polyline points="14 2 14 8 20 8"/></svg>
              </div>
              <div><div class="fc-title">Contrat Validé</div><div class="fc-sub">✓ Signature électronique</div></div>
            </div>
          </div>
        </div>
      </div>
    </div>
  </div>
</section>

<!-- KEY STATS SECTION -->
<section id="stats">
  <div class="container">
    <div class="stats-grid">
      <div class="stat-card card-hover">
        <div class="stat-icon" style="background:#DBEAFE;">
          <svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="#2563EB" stroke-width="2" stroke-linecap="round"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/><polyline points="14 2 14 8 20 8"/></svg>
        </div>
        <div class="stat-value" style="color:#2563EB;">320+</div>
        <div class="stat-label">Demandes Traitées</div>
      </div>
      <div class="stat-card card-hover">
        <div class="stat-icon" style="background:#D1FAE5;">
          <svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="#22C55E" stroke-width="2" stroke-linecap="round"><rect x="1" y="4" width="22" height="16" rx="2"/><line x1="1" y1="10" x2="23" y2="10"/></svg>
        </div>
        <div class="stat-value" style="color:#22C55E;">98%</div>
        <div class="stat-label">Paiements Réussis</div>
      </div>
      <div class="stat-card card-hover">
        <div class="stat-icon" style="background:#FEF3C7;">
          <svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="#F59E0B" stroke-width="2" stroke-linecap="round"><circle cx="12" cy="12" r="10"/><polyline points="12 6 12 12 16 14"/></svg>
        </div>
        <div class="stat-value" style="color:#F59E0B;">48h</div>
        <div class="stat-label">Temps Moyen de Validation</div>
      </div>
      <div class="stat-card card-hover">
        <div class="stat-icon" style="background:#FEE2E2;">
          <svg width="24" height="24" viewBox="0 0 24 24" fill="#EF4444"><path d="M12 2l3.09 6.26L22 9.27l-5 4.87 1.18 6.88L12 17.77l-6.18 3.25L7 14.14 2 9.27l6.91-1.01L12 2z"/></svg>
        </div>
        <div class="stat-value" style="color:#EF4444;">96%</div>
        <div class="stat-label">Satisfaction Usagers</div>
      </div>
    </div>
  </div>
</section>

<!-- HOW IT WORKS -->
<section id="how">
  <div class="container">
    <div style="text-align:center; margin-bottom:64px;">
      <div class="badge" style="margin: 0 auto 16px;">PROCESSUS SIMPLIFIÉ</div>
      <h2 class="section-title">Comment ça fonctionne ?</h2>
      <p class="section-sub" style="margin: 0 auto;">De la soumission du dossier à l'occupation de votre espace commercial.</p>
    </div>
    <div class="timeline">
      <div class="step"><div class="step-icon" style="background:#DBEAFE;"><svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="#2563EB" stroke-width="2" stroke-linecap="round"><path d="M17 21v-2a4 4 0 0 0-4-4H5a4 4 0 0 0-4 4v2"/><circle cx="9" cy="7" r="4"/></svg><span class="step-num" style="background:#2563EB;">01</span></div><div class="step-t">Inscription</div><div class="step-d">Créer un compte usager</div></div>
      <div class="step"><div class="step-icon" style="background:#EDE9FE;"><svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="#7C3AED" stroke-width="2" stroke-linecap="round"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/><polyline points="14 2 14 8 20 8"/></svg><span class="step-num" style="background:#7C3AED;">02</span></div><div class="step-t">Demande</div><div class="step-d">Soumettre votre projet</div></div>
      <div class="step"><div class="step-icon" style="background:#CCFBF1;"><svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="#0F766E" stroke-width="2" stroke-linecap="round"><path d="M12 2L2 7l10 5 10-5-10-5z"/></svg><span class="step-num" style="background:#0F766E;">03</span></div><div class="step-t">Contrôle IA</div><div class="step-d">Vérification automatique</div></div>
      <div class="step"><div class="step-icon" style="background:#FEF3C7;"><svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="#B45309" stroke-width="2" stroke-linecap="round"><path d="M1 12s4-8 11-8 11 8 11 8-4 8-11 8-11-8-11-8z"/><circle cx="12" cy="12" r="3"/></svg><span class="step-num" style="background:#B45309;">04</span></div><div class="step-t">Revue CROUS</div><div class="step-d">Examen administratif</div></div>
      <div class="step"><div class="step-icon" style="background:#D1FAE5;"><svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="#22C55E" stroke-width="2" stroke-linecap="round"><polyline points="20 6 9 17 4 12"/></svg><span class="step-num" style="background:#22C55E;">05</span></div><div class="step-t">Approbation</div><div class="step-d">Notification instantanée</div></div>
      <div class="step"><div class="step-icon" style="background:#FEE2E2;"><svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="#DC2626" stroke-width="2" stroke-linecap="round"><rect x="3" y="11" width="18" height="11" rx="2"/><path d="M7 11V7a5 5 0 0 1 10 0v4"/></svg><span class="step-num" style="background:#DC2626;">06</span></div><div class="step-t">Contrat</div><div class="step-d">Signature électronique</div></div>
      <div class="step"><div class="step-icon" style="background:#E0F2FE;"><svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="#0369A1" stroke-width="2" stroke-linecap="round"><rect x="1" y="4" width="22" height="16" rx="2"/></svg><span class="step-num" style="background:#0369A1;">07</span></div><div class="step-t">Paiement</div><div class="step-d">Orange Money / Wave</div></div>
      <div class="step"><div class="step-icon" style="background:#DBEAFE;"><svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="#2563EB" stroke-width="2" stroke-linecap="round"><path d="M3 9l9-7 9 7v11a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2z"/></svg><span class="step-num" style="background:#2563EB;">08</span></div><div class="step-t">Occupation</div><div class="step-d">Remise des clés</div></div>
    </div>
  </div>
</section>

<!-- SERVICES & FEATURES -->
<section id="features">
  <div class="container">
    <div style="text-align:center; margin-bottom:64px;">
      <div class="badge" style="margin: 0 auto 16px;">FONCTIONNALITÉS</div>
      <h2 class="section-title">Services Intégrés du Campus</h2>
      <p class="section-sub" style="margin: 0 auto;">Tout ce dont vous avez besoin pour gérer et exploiter un commerce universitaire.</p>
    </div>
    <div class="features-grid">
      <div class="feat-card card-hover" style="background:linear-gradient(135deg, #DBEAFE, #EFF6FF);" onclick="openDashboardPortal('demandes')">
        <div class="feat-icon"><svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="#2563EB" stroke-width="2" stroke-linecap="round"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/><polyline points="14 2 14 8 20 8"/></svg></div>
        <div class="feat-title">Demandes en Ligne</div>
        <div class="feat-desc">Soumettez votre projet commercial en quelques clics avec pièces justificatives.</div>
      </div>
      <div class="feat-card card-hover" style="background:linear-gradient(135deg, #EDE9FE, #F5F3FF);" onclick="openDashboardPortal('demandes')">
        <div class="feat-icon"><svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="#7C3AED" stroke-width="2" stroke-linecap="round"><polyline points="23 6 13.5 15.5 8.5 10.5 1 18"/></svg></div>
        <div class="feat-title">Suivi en Temps Réel</div>
        <div class="feat-desc">Recevez des notifications par SMS & Email à chaque étape d'instruction.</div>
      </div>
      <div class="feat-card card-hover" style="background:linear-gradient(135deg, #CCFBF1, #F0FDF4);" onclick="openDashboardPortal('contrats')">
        <div class="feat-icon"><svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="#0F766E" stroke-width="2" stroke-linecap="round"><rect x="3" y="11" width="18" height="11" rx="2"/><path d="M7 11V7a5 5 0 0 1 10 0v4"/></svg></div>
        <div class="feat-title">Contrats Électroniques</div>
        <div class="feat-desc">Signature légale certifiée conforme aux lois de la République du Sénégal.</div>
      </div>
      <div class="feat-card card-hover" style="background:linear-gradient(135deg, #FEF3C7, #FFFBEB);" onclick="openDashboardPortal('paiements')">
        <div class="feat-icon"><svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="#B45309" stroke-width="2" stroke-linecap="round"><rect x="1" y="4" width="22" height="16" rx="2"/><line x1="1" y1="10" x2="23" y2="10"/></svg></div>
        <div class="feat-title">Paiements Sécurisés</div>
        <div class="feat-desc">Réglez vos redevances via Wave, Orange Money ou carte bancaire avec reçus automatiques.</div>
      </div>
      <div class="feat-card card-hover" style="background:linear-gradient(135deg, #FEE2E2, #FFF5F5);" onclick="openDashboardPortal('incidents')">
        <div class="feat-icon"><svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="#DC2626" stroke-width="2" stroke-linecap="round"><path d="M18 8A6 6 0 0 0 6 8c0 7-3 9-3 9h18s-3-2-3-9"/><path d="M13.73 21a2 2 0 0 1-3.46 0"/></svg></div>
        <div class="feat-title">Signalement d'Incidents</div>
        <div class="feat-desc">Signalez les pannes techniques ou besoins de maintenance pour intervention sous 72h.</div>
      </div>
      <div class="feat-card card-hover" style="background:linear-gradient(135deg, #E0F2FE, #F0F9FF);" onclick="openDashboardPortal('admin')">
        <div class="feat-icon"><svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="#0369A1" stroke-width="2" stroke-linecap="round"><path d="M12 2L2 7l10 5 10-5-10-5z"/><path d="M2 17l10 5 10-5"/><path d="M2 12l10 5 10-5"/></svg></div>
        <div class="feat-title">Gestion Administrateur</div>
        <div class="feat-desc">Un espace d'administration dédié pour valider et piloter le patrimoine commercial.</div>
      </div>
    </div>
  </div>
</section>

<!-- INTERACTIVE CAMPUS MAP -->
<section id="map">
  <div class="container">
    <div class="map-layout">
      <div>
        <div class="badge" style="margin-bottom:16px;">PLAN EN DIRECT</div>
        <h2 class="section-title">Locaux du Campus de Thiès</h2>
        <p class="section-sub">Consultez la disponibilité des emplacements commerciaux sur la carte interactive du campus.</p>

        <div class="map-legend">
          <div class="leg-item"><div class="leg-dot" style="background:#22C55E;"></div> Local Disponible (Faire une demande)</div>
          <div class="leg-item"><div class="leg-dot" style="background:#2563EB;"></div> Local Occupé (En souscription)</div>
          <div class="leg-item"><div class="leg-dot" style="background:#F59E0B;"></div> En Maintenance / Travaux</div>
        </div>

        <div class="map-detail" id="map-detail">
          <div class="map-detail-head">
            <span class="map-dname" id="d-name">Local A1</span>
            <span class="map-tag" id="d-tag">Disponible</span>
          </div>
          <div class="map-dcat" id="d-cat">Catégorie : Services & Papeterie</div>
          <div style="font-weight:700; color:#2563EB; font-size:16px; margin-bottom:16px;" id="d-price">50 000 FCFA / mois</div>
          <button class="btn-req" id="d-btn" onclick="openDashboardPortal('demandes')">Déposer un dossier pour ce local</button>
        </div>

        <button class="btn-exp-map" onclick="openDashboardPortal('locaux')">
          <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="2" stroke-linecap="round"><polygon points="1 6 1 22 8 18 16 22 23 18 23 2 16 6 8 2 1 6"/><line x1="8" y1="2" x2="8" y2="18"/><line x1="16" y1="6" x2="16" y2="22"/></svg>
          Ouvrir la Liste des Locaux
        </button>
      </div>

      <div class="campus-svg-wrap" id="campus-map">
        <svg viewBox="0 0 600 440" xmlns="http://www.w3.org/2000/svg" width="100%" height="100%">
          <defs>
            <pattern id="cgrid" width="40" height="40" patternUnits="userSpaceOnUse">
              <path d="M 40 0 L 0 0 0 40" fill="none" stroke="rgba(37,99,235,0.08)" stroke-width="1"/>
            </pattern>
          </defs>
          <rect width="600" height="440" fill="url(#cgrid)"/>
          <line x1="240" y1="0" x2="240" y2="440" stroke="rgba(148,163,184,0.4)" stroke-width="3"/>
          <line x1="0" y1="180" x2="600" y2="180" stroke="rgba(148,163,184,0.4)" stroke-width="3"/>

          <rect x="16" y="16" width="200" height="140" rx="14" fill="#BFDBFE" opacity="0.65"/>
          <text x="116" y="86" text-anchor="middle" font-family="Inter,sans-serif" font-size="12" font-weight="700" fill="#1E3A8A">Bâtiment Admin CROUS</text>

          <rect x="260" y="16" width="320" height="140" rx="14" fill="#BBF7D0" opacity="0.65"/>
          <text x="420" y="86" text-anchor="middle" font-family="Inter,sans-serif" font-size="12" font-weight="700" fill="#065F46">Faculté des Sciences & Tech</text>

          <rect x="16" y="210" width="200" height="210" rx="14" fill="#FDE68A" opacity="0.65"/>
          <text x="116" y="315" text-anchor="middle" font-family="Inter,sans-serif" font-size="12" font-weight="700" fill="#92400E">Résidences Etudiantes</text>

          <rect x="260" y="210" width="320" height="210" rx="14" fill="#FCA5A5" opacity="0.45"/>
          <text x="420" y="315" text-anchor="middle" font-family="Inter,sans-serif" font-size="12" font-weight="700" fill="#991B1B">Zone Commerciale Principale</text>
        </svg>
      </div>
    </div>
  </div>
</section>

<!-- CATEGORIES SECTION -->
<section id="categories">
  <div class="container">
    <div style="text-align:center; margin-bottom:64px;">
      <div class="badge" style="margin: 0 auto 16px;">CATÉGORIES D'ACTIVITÉS</div>
      <h2 class="section-title">Espaces par Type de Commerce</h2>
      <p class="section-sub" style="margin: 0 auto;">Découvrez les secteurs d'activités autorisés sur le campus universitaire.</p>
    </div>

    <div class="cats-scroll">
      <div class="cat-card card-hover" style="background:linear-gradient(135deg, #FEF3C7, #FDE68A);" onclick="openDashboardPortal('demandes')">
        <div class="cat-emoji">📚</div>
        <div class="cat-name">Papeterie</div>
        <div class="cat-desc">Fournitures scolaires, livres, travaux d'impression</div>
        <div class="cat-foot">
          <span class="cat-count" style="color:#F59E0B;">8 locaux</span>
          <span class="btn-expl" style="color:#F59E0B;">Soumettre</span>
        </div>
      </div>

      <div class="cat-card card-hover" style="background:linear-gradient(135deg, #D1FAE5, #A7F3D0);" onclick="openDashboardPortal('demandes')">
        <div class="cat-emoji">🥗</div>
        <div class="cat-name">Alimentation</div>
        <div class="cat-desc">Restauration rapide, kiosques snack, supérette</div>
        <div class="cat-foot">
          <span class="cat-count" style="color:#22C55E;">12 locaux</span>
          <span class="btn-expl" style="color:#22C55E;">Soumettre</span>
        </div>
      </div>

      <div class="cat-card card-hover" style="background:linear-gradient(135deg, #FCE7F3, #FBCFE8);" onclick="openDashboardPortal('demandes')">
        <div class="cat-emoji">💄</div>
        <div class="cat-name">Cosmétiques</div>
        <div class="cat-desc">Produits de soins, salon de coiffure et esthétique</div>
        <div class="cat-foot">
          <span class="cat-count" style="color:#EC4899;">6 locaux</span>
          <span class="btn-expl" style="color:#EC4899;">Soumettre</span>
        </div>
      </div>

      <div class="cat-card card-hover" style="background:linear-gradient(135deg, #DBEAFE, #BFDBFE);" onclick="openDashboardPortal('demandes')">
        <div class="cat-emoji">💻</div>
        <div class="cat-name">Électronique</div>
        <div class="cat-desc">Maintenance informatique, vente accessoires mobile</div>
        <div class="cat-foot">
          <span class="cat-count" style="color:#2563EB;">4 locaux</span>
          <span class="btn-expl" style="color:#2563EB;">Soumettre</span>
        </div>
      </div>

      <div class="cat-card card-hover" style="background:linear-gradient(135deg, #EDE9FE, #DDD6FE);" onclick="openDashboardPortal('demandes')">
        <div class="cat-emoji">🛠️</div>
        <div class="cat-name">Multiservices</div>
        <div class="cat-desc">Transfert d'argent, pressing, couture, cordonnerie</div>
        <div class="cat-foot">
          <span class="cat-count" style="color:#7C3AED;">10 locaux</span>
          <span class="btn-expl" style="color:#7C3AED;">Soumettre</span>
        </div>
      </div>
    </div>
  </div>
</section>

<!-- TESTIMONIALS -->
<section id="testimonials">
  <div class="container">
    <div style="text-align:center; margin-bottom:64px;">
      <div class="badge" style="margin: 0 auto 16px;">TÉMOIGNAGES</div>
      <h2 class="section-title">Ce qu'en disent nos usagers</h2>
      <p class="section-sub" style="margin: 0 auto;">Étudiants entrepreneurs et commerçants partagent leur expérience avec KaayJob.</p>
    </div>
    <div class="testi-grid">
      <div class="testi card-hover">
        <div class="testi-stars">★★★★★</div>
        <p class="testi-text">"Grâce à la plateforme KaayJob, j'ai pu obtenir l'attribution de mon kiosque de photocopie sans faire la queue. Le paiement par Wave est hyper fluide."</p>
        <div class="testi-person">
          <div class="tav" style="background:linear-gradient(135deg,#2563EB,#1D4ED8);">AD</div>
          <div>
            <div class="tname">Aminata Diallo</div>
            <div class="trole">Étudiante Entrepreneur</div>
            <div class="tuni">UIDT Thiès</div>
          </div>
        </div>
      </div>

      <div class="testi card-hover">
        <div class="testi-stars">★★★★★</div>
        <p class="testi-text">"La signature électronique du contrat et le suivi des demandes en direct m'ont fait gagner un temps précieux. Tout est clair et transparent."</p>
        <div class="testi-person">
          <div class="tav" style="background:linear-gradient(135deg,#059669,#047857);">IN</div>
          <div>
            <div class="tname">Ibrahima Ndiaye</div>
            <div class="trole">Gérant Multimédia</div>
            <div class="tuni">Espace Campus</div>
          </div>
        </div>
      </div>

      <div class="testi card-hover">
        <div class="testi-stars">★★★★★</div>
        <p class="testi-text">"En tant qu'administratrice CROUS, cette application a réduit les délais de traitement des dossiers de 70% et sécurisé les recettes des loyers."</p>
        <div class="testi-person">
          <div class="tav" style="background:linear-gradient(135deg,#7C3AED,#6D28D9);">KS</div>
          <div>
            <div class="tname">Dr. Khady Sall</div>
            <div class="trole">Responsable Gestion Immobilière</div>
            <div class="tuni">CROUS Thiès</div>
          </div>
        </div>
      </div>
    </div>
  </div>
</section>

<!-- FAQ SECTION -->
<section id="faq">
  <div class="container">
    <div style="text-align:center; margin-bottom:64px;">
      <div class="badge" style="margin: 0 auto 16px;">FAQ</div>
      <h2 class="section-title">Foire Aux Questions</h2>
      <p class="section-sub" style="margin: 0 auto;">Retrouvez les réponses aux questions les plus fréquemment posées.</p>
    </div>

    <div class="faq-list">
      <div class="faq-item open">
        <button class="faq-q">
          Comment déposer une demande de local commercial ?
          <div class="faq-tog"><svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="#475569" stroke-width="2.5" stroke-linecap="round"><polyline points="6 9 12 15 18 9"/></svg></div>
        </button>
        <div class="faq-ans"><div class="faq-ans-in">Cliquez sur "Accéder au Portail", naviguez dans l'onglet "Mes Demandes", remplissez les détails de votre commerce et téléchargez la pièce d'identité et votre justificatif étudiant ou professionnel.</div></div>
      </div>

      <div class="faq-item">
        <button class="faq-q">
          Quels sont les moyens de paiement acceptés pour le loyer ?
          <div class="faq-tog"><svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="#475569" stroke-width="2.5" stroke-linecap="round"><polyline points="6 9 12 15 18 9"/></svg></div>
        </button>
        <div class="faq-ans"><div class="faq-ans-in">Vous pouvez payer mensuellement via Orange Money, Wave ou par carte bancaire. Un reçu numérique avec QR code de vérification est généré instantanément.</div></div>
      </div>

      <div class="faq-item">
        <button class="faq-q">
          Comment fonctionne le signalement des incidents techniques ?
          <div class="faq-tog"><svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="#475569" stroke-width="2.5" stroke-linecap="round"><polyline points="6 9 12 15 18 9"/></svg></div>
        </button>
        <div class="faq-ans"><div class="faq-ans-in">Depuis votre tableau de bord, vous pouvez ouvrir l'onglet "Signalements" pour notifier la régie technique du CROUS d'un problème d'eau, d'électricité ou de structure.</div></div>
      </div>
    </div>
  </div>
</section>

<!-- CALL TO ACTION -->
<section id="cta">
  <div class="container">
    <div class="cta-box">
      <div class="cg1"></div><div class="cg2"></div>
      <div class="cta-in">
        <div class="cta-tag">🚀 Plateforme En Ligne</div>
        <h2 class="cta-title">Digitalisez la gestion de vos espaces dès aujourd'hui</h2>
        <p class="cta-sub">Rejoignez des centaines de commerçants et la direction du CROUS sur la plateforme de référence du campus de Thiès.</p>
        <div class="cta-btns">
          <button class="btn-cta-w" onclick="openDashboardPortal()">Créer un compte usager</button>
          <button class="btn-cta-g" onclick="openDashboardPortal('admin')">Accès Administration</button>
        </div>
      </div>
    </div>
  </div>
</section>

<!-- FOOTER -->
<footer id="footer">
  <div class="container">
    <div class="footer-grid">
      <div>
        <a href="#" class="nav-logo">
          <div class="nav-logo-icon">KJ</div>
          <span class="nav-logo-text">Kaay<span>Job</span></span>
        </a>
        <p class="footer-desc">Système officiel de gestion du patrimoine commercial du Centre Régional des Œuvres Universitaires de Thiès (CROUS-T).</p>
      </div>
      <div class="fcol">
        <h4>Navigation</h4>
        <ul>
          <li><a href="#hero">Accueil</a></li>
          <li><a href="#features">Services</a></li>
          <li><a href="#map">Locaux Campus</a></li>
          <li><a href="#categories">Catégories</a></li>
        </ul>
      </div>
      <div class="fcol">
        <h4>Portail</h4>
        <ul>
          <li><a href="javascript:void(0)" onclick="openDashboardPortal('demandes')">Mes Demandes</a></li>
          <li><a href="javascript:void(0)" onclick="openDashboardPortal('contrats')">Contrats</a></li>
          <li><a href="javascript:void(0)" onclick="openDashboardPortal('paiements')">Paiements Loyer</a></li>
          <li><a href="javascript:void(0)" onclick="openDashboardPortal('admin')">Zone Admin</a></li>
        </ul>
      </div>
      <div class="fcol">
        <h4>Support</h4>
        <ul>
          <li><a href="#faq">Centre d'aide FAQ</a></li>
          <li><a href="javascript:void(0)" onclick="openDashboardPortal('incidents')">Signalements</a></li>
          <li><a href="#">Réglementation CROUS</a></li>
        </ul>
      </div>
      <div class="fcol">
        <h4>Mentions</h4>
        <ul>
          <li><a href="#">Confidentialité</a></li>
          <li><a href="#">Conditions de bail</a></li>
          <li><a href="#">Légal Senelex</a></li>
        </ul>
      </div>
    </div>

    <div class="foot-contacts">
      <div class="fcontact"><div class="fci">✉</div><span class="fct">contact@kaayjob.sn</span></div>
      <div class="fcontact"><div class="fci">📞</div><span class="fct">+221 76 742 21 24</span></div>
      <div class="fcontact"><div class="fci">📍</div><span class="fct">Campus UIDT, Thiès, Sénégal</span></div>
    </div>

    <div class="foot-bottom">
      <span class="foot-copy">© 2026 KaayJob THIES. Développé pour le CROUS de Thiès. Tous droits réservés.</span>
      <div class="status-pill"><div class="sdot"></div> Système 100% Opérationnel</div>
    </div>
  </div>
</footer>

<!-- DASHBOARD PORTAL / DRAWER MODAL (INTEGRATING FIGMA / TAILWIND SPECIFICATIONS) -->
<div class="drawer-overlay" id="dashboard-drawer">
  <div class="dashboard-modal">
    <!-- DARK SIDEBAR USING SPECIFIED COLORS (--color-sidebar: #0F172A) -->
    <aside class="app-sidebar">
      <div>
        <div class="sidebar-header">
          <div class="sidebar-logo-icon">KJ</div>
          <div class="sidebar-logo-title">Kaay<span>Job</span></div>
        </div>

        <ul class="sidebar-menu">
          <li class="sidebar-item active" onclick="switchTab('dashboard')">
            <svg fill="none" viewBox="0 0 24 24"><path d="M3 13h8V3H3v10zm0 8h8v-6H3v6zm10 0h8V11h-8v10zm0-18v6h8V3h-8z" stroke-width="2" stroke-linecap="round"/></svg>
            Tableau de Bord
          </li>
          <li class="sidebar-item" onclick="switchTab('demandes')">
            <svg fill="none" viewBox="0 0 24 24"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z" stroke-width="2" stroke-linecap="round"/><polyline points="14 2 14 8 20 8"/></svg>
            Mes Demandes
          </li>
          <li class="sidebar-item" onclick="switchTab('locaux')">
            <svg fill="none" viewBox="0 0 24 24"><path d="M3 9l9-7 9 7v11a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2z" stroke-width="2" stroke-linecap="round"/></svg>
            Locaux du Campus
          </li>
          <li class="sidebar-item" onclick="switchTab('contrats')">
            <svg fill="none" viewBox="0 0 24 24"><path d="M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10z" stroke-width="2" stroke-linecap="round"/></svg>
            Contrats & Signatures
          </li>
          <li class="sidebar-item" onclick="switchTab('paiements')">
            <svg fill="none" viewBox="0 0 24 24"><rect x="1" y="4" width="22" height="16" rx="2" stroke-width="2"/><line x1="1" y1="10" x2="23" y2="10" stroke-width="2"/></svg>
            Paiements Loyer
          </li>
          <li class="sidebar-item" onclick="switchTab('incidents')">
            <svg fill="none" viewBox="0 0 24 24"><path d="M10.29 3.86L1.82 18a2 2 0 0 0 1.71 3h16.94a2 2 0 0 0 1.71-3L13.71 3.86a2 2 0 0 0-3.42 0z" stroke-width="2"/><line x1="12" y1="9" x2="12" y2="13" stroke-width="2"/><line x1="12" y1="17" x2="12.01" y2="17" stroke-width="2"/></svg>
            Signalement Incidents
          </li>
          <li class="sidebar-item" onclick="switchTab('admin')" style="margin-top:12px; border-top:1px solid #1E293B; padding-top:16px;">
            <svg fill="none" viewBox="0 0 24 24"><path d="M12 15a3 3 0 1 0 0-6 3 3 0 0 0 0 6z" stroke-width="2"/><path d="M19.4 15a1.65 1.65 0 0 0 .33 1.82l.06.06a2 2 0 0 1 0 2.83 2 2 0 0 1-2.83 0l-.06-.06a1.65 1.65 0 0 0-1.82-.33 1.65 1.65 0 0 0-1 1.51V21a2 2 0 0 1-2 2 2 2 0 0 1-2-2v-.09A1.65 1.65 0 0 0 9 19.4a1.65 1.65 0 0 0-1.82.33l-.06.06a2 2 0 0 1-2.83 0 2 2 0 0 1 0-2.83l.06-.06a1.65 1.65 0 0 0 .33-1.82 1.65 1.65 0 0 0-1.51-1H3a2 2 0 0 1-2-2 2 2 0 0 1 2-2h.09A1.65 1.65 0 0 0 4.6 9a1.65 1.65 0 0 0-.33-1.82l-.06-.06a2 2 0 0 1 0-2.83 2 2 0 0 1 2.83 0l.06.06a1.65 1.65 0 0 0 1.82.33H9a1.65 1.65 0 0 0 1-1.51V3a2 2 0 0 1 2-2 2 2 0 0 1 2 2v.09a1.65 1.65 0 0 0 1 1.51 1.65 1.65 0 0 0 1.82-.33l.06-.06a2 2 0 0 1 2.83 0 2 2 0 0 1 0 2.83l-.06.06a1.65 1.65 0 0 0-.33 1.82V9a1.65 1.65 0 0 0 1.51 1H21a2 2 0 0 1 2 2 2 2 0 0 1-2 2h-.09a1.65 1.65 0 0 0-1.51 1z" stroke-width="2"/></svg>
            Espace Admin CROUS
          </li>
        </ul>
      </div>

      <div class="user-profile-card">
        <div class="u-avatar" id="portal-avatar">MD</div>
        <div class="u-info">
          <div class="u-name" id="portal-user-name">Moussa Diallo</div>
          <div class="u-role" id="portal-user-role">Usager / Commerçant</div>
        </div>
      </div>
    </aside>

    <!-- DASHBOARD MAIN CONTENT AREA -->
    <main class="app-main">
      <div class="app-topbar">
        <h2 class="topbar-title" id="tab-title">Tableau de Bord</h2>
        <div class="topbar-actions">
          <button class="btn-ghost" style="border:1px solid #E2E8F0; font-size:13px;" onclick="toggleUserRole()">
            🔀 Permuter Rôle (Usager / Admin)
          </button>
          <button class="btn-close-drawer" onclick="closeDashboardPortal()" title="Fermer">✕</button>
        </div>
      </div>

      <div class="app-content">

        <!-- TAB 1: OVERVIEW DASHBOARD -->
        <div class="tab-content active" id="tab-dashboard">
          <div style="display:grid; grid-template-columns: repeat(3, 1fr); gap: 20px; margin-bottom: 28px;">
            <div style="background:white; padding:24px; border-radius:18px; border:1px solid #E2E8F0; box-shadow:0 2px 10px rgba(0,0,0,0.02);">
              <div style="font-size:13px; color:#64748B; margin-bottom:6px;">Demande en Cours</div>
              <div style="font-size:28px; font-weight:800; color:#2563EB;" id="dash-stat-req">1 Active</div>
              <div style="font-size:12px; color:#22C55E; margin-top:6px; font-weight:600;">Dossier sous vérification IA</div>
            </div>

            <div style="background:white; padding:24px; border-radius:18px; border:1px solid #E2E8F0; box-shadow:0 2px 10px rgba(0,0,0,0.02);">
              <div style="font-size:13px; color:#64748B; margin-bottom:6px;">Local Attribué</div>
              <div style="font-size:28px; font-weight:800; color:#0F172A;">Local C-204</div>
              <div style="font-size:12px; color:#64748B; margin-top:6px;">Zone Papeterie & Services</div>
            </div>

            <div style="background:white; padding:24px; border-radius:18px; border:1px solid #E2E8F0; box-shadow:0 2px 10px rgba(0,0,0,0.02);">
              <div style="font-size:13px; color:#64748B; margin-bottom:6px;">Prochain Loyer</div>
              <div style="font-size:28px; font-weight:800; color:#7C3AED;">50 000 FCFA</div>
              <div style="font-size:12px; color:#F59E0B; margin-top:6px; font-weight:600;">Échéance : 05 du mois</div>
            </div>
          </div>

          <div class="table-container">
            <div style="padding:20px; border-bottom:1px solid #E2E8F0; font-weight:700; font-family:var(--font-heading); color:#0F172A;">Historique Récent des Activités</div>
            <table class="data-table">
              <thead>
                <tr>
                  <th>Référence</th>
                  <th>Type de Commerce</th>
                  <th>Emplacement</th>
                  <th>Date</th>
                  <th>Statut</th>
                </tr>
              </thead>
              <tbody id="dash-activities-body">
                <tr>
                  <td><strong>#REQ-2026-089</strong></td>
                  <td>Papeterie & Photocopie</td>
                  <td>Local C-204 (Faculté)</td>
                  <td>09/08/2026</td>
                  <td><span class="status-badge pending">● En cours d'examen</span></td>
                </tr>
                <tr>
                  <td><strong>#PAY-2026-041</strong></td>
                  <td>Loyer Mois d'Août</td>
                  <td>Kiosque B12 (Résidences)</td>
                  <td>01/08/2026</td>
                  <td><span class="status-badge approved">✓ Payé (Wave)</span></td>
                </tr>
              </tbody>
            </table>
          </div>
        </div>

        <!-- TAB 2: MES DEMANDES DE LOCAL -->
        <div class="tab-content" id="tab-demandes">
          <div style="display:grid; grid-template-columns: 1fr 1.2fr; gap: 32px;">
            <div style="background:white; padding:28px; border-radius:20px; border:1px solid #E2E8F0;">
              <h3 style="font-family:var(--font-heading); font-size:18px; font-weight:700; margin-bottom:20px; color:#0F172A;">Nouvelle Demande de Local Commercial</h3>
              <form id="form-new-request" onsubmit="handleNewRequest(event)">
                <div class="form-group">
                  <label class="form-label">Nom complet du demandeur</label>
                  <input type="text" class="form-input" id="req-name" value="Moussa Diallo" required />
                </div>
                <div class="form-group">
                  <label class="form-label">Téléphone (WhatsApp / Wave)</label>
                  <input type="tel" class="form-input" id="req-phone" value="+221 77 123 45 67" required />
                </div>
                <div class="form-group">
                  <label class="form-label">Catégorie d'Activité</label>
                  <select class="form-select" id="req-category" required>
                    <option value="Papeterie">Papeterie / Imprimerie</option>
                    <option value="Alimentation">Alimentation / Kiosque Snack</option>
                    <option value="Cosmétiques">Cosmétiques & Coiffure</option>
                    <option value="Électronique">Maintenance Électronique</option>
                    <option value="Multiservices">Multiservices / Transfert d'argent</option>
                  </select>
                </div>
                <div class="form-group">
                  <label class="form-label">Local souhaite</label>
                  <select class="form-select" id="req-location" required>
                    <option value="Local A1 - Zone Admin">Local A1 (Bâtiment Admin)</option>
                    <option value="Local C-204 - Faculté">Local C-204 (Faculté Sciences)</option>
                    <option value="Kiosque B12 - Résidences">Kiosque B12 (Résidences)</option>
                    <option value="Espace E3 - Commercial">Espace E3 (Zone Commerciale)</option>
                  </select>
                </div>
                <div class="form-group">
                  <label class="form-label">Description du projet</label>
                  <textarea class="form-textarea" id="req-desc" rows="3" placeholder="Présentez brièvement votre activité commerciale..." required></textarea>
                </div>
                <button type="submit" class="btn-primary" style="width:100%; justify-content:center; padding:14px;">Soumettre la Demande</button>
              </form>
            </div>

            <div>
              <h3 style="font-family:var(--font-heading); font-size:18px; font-weight:700; margin-bottom:20px; color:#0F172A;">Vos Demandes Soumises</h3>
              <div class="table-container">
                <table class="data-table">
                  <thead>
                    <tr>
                      <th>Ref</th>
                      <th>Catégorie</th>
                      <th>Local</th>
                      <th>Statut</th>
                    </tr>
                  </thead>
                  <tbody id="user-requests-list">
                    <tr>
                      <td>#REQ-891</td>
                      <td>Papeterie</td>
                      <td>Local C-204</td>
                      <td><span class="status-badge pending">● En Examen</span></td>
                    </tr>
                  </tbody>
                </table>
              </div>
            </div>
          </div>
        </div>

        <!-- TAB 3: LOCAUX DU CAMPUS -->
        <div class="tab-content" id="tab-locaux">
          <div style="display:grid; grid-template-columns: repeat(3, 1fr); gap: 20px;" id="locaux-grid">
            <div style="background:white; border-radius:18px; padding:20px; border:1px solid #E2E8F0;">
              <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:12px;">
                <span style="font-weight:700; color:#0F172A;">Local A1</span>
                <span class="status-badge approved">Disponible</span>
              </div>
              <div style="font-size:13px; color:#64748B; margin-bottom:16px;">Bâtiment Administrateur · 25 m²</div>
              <div style="font-size:18px; font-weight:800; color:#2563EB; margin-bottom:16px;">60 000 FCFA / mois</div>
              <button class="btn-primary" style="width:100%; justify-content:center;" onclick="openDashboardPortal('demandes')">Choisir ce Local</button>
            </div>

            <div style="background:white; border-radius:18px; padding:20px; border:1px solid #E2E8F0;">
              <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:12px;">
                <span style="font-weight:700; color:#0F172A;">Kiosque B12</span>
                <span class="status-badge rejected">Occupé</span>
              </div>
              <div style="font-size:13px; color:#64748B; margin-bottom:16px;">Résidences Étudiantes · 15 m²</div>
              <div style="font-size:18px; font-weight:800; color:#0F172A; margin-bottom:16px;">45 000 FCFA / mois</div>
              <button class="btn-ghost" style="width:100%; border:1px solid #CBD5E1; color:#94A3B8;" disabled>Indisponible</button>
            </div>

            <div style="background:white; border-radius:18px; padding:20px; border:1px solid #E2E8F0;">
              <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:12px;">
                <span style="font-weight:700; color:#0F172A;">Espace C-10</span>
                <span class="status-badge approved">Disponible</span>
              </div>
              <div style="font-size:13px; color:#64748B; margin-bottom:16px;">Zone Commerciale · 30 m²</div>
              <div style="font-size:18px; font-weight:800; color:#2563EB; margin-bottom:16px;">75 000 FCFA / mois</div>
              <button class="btn-primary" style="width:100%; justify-content:center;" onclick="openDashboardPortal('demandes')">Choisir ce Local</button>
            </div>
          </div>
        </div>

        <!-- TAB 4: CONTRATS & SIGNATURES -->
        <div class="tab-content" id="tab-contrats">
          <div style="background:white; padding:32px; border-radius:20px; border:1px solid #E2E8F0; max-width:720px; margin:0 auto;">
            <h3 style="font-family:var(--font-heading); font-size:20px; font-weight:700; margin-bottom:16px; color:#0F172A;">Bail Commercial D'Occupation — CROUS Thiès</h3>
            <p style="font-size:14px; color:#475569; line-height:1.7; margin-bottom:20px;">
              Entre le <strong>Centre Régional des Œuvres Universitaires de Thiès (CROUS-T)</strong> et l'occupant <strong>Moussa Diallo</strong>, il est convenu la mise à disposition de l'espace commercial <strong>Local C-204</strong> pour une redevance mensuelle de 50 000 FCFA.
            </p>
            <div style="background:#F8FAFC; padding:16px; border-radius:12px; border:1px dashed #CBD5E1; margin-bottom:24px; font-size:13px; color:#64748B;">
              <strong>Certificat numérique de signature :</strong> Conforme aux dispositions de la loi sénégalaise sur les transactions électroniques.
            </div>

            <div style="display:flex; align-items:center; justify-content:space-between;">
              <div id="contract-status-text" style="font-weight:700; color:#F59E0B;">● En attente de signature</div>
              <button class="btn-primary" id="btn-sign-contract" onclick="signContract()">Signer Numériquement le Contrat</button>
            </div>
          </div>
        </div>

        <!-- TAB 5: PAIEMENTS LOYER -->
        <div class="tab-content" id="tab-paiements">
          <div style="display:grid; grid-template-columns: 1.2fr 1fr; gap: 32px;">
            <div style="background:white; padding:28px; border-radius:20px; border:1px solid #E2E8F0;">
              <h3 style="font-family:var(--font-heading); font-size:18px; font-weight:700; margin-bottom:20px; color:#0F172A;">Règlement de la Redevance Mensuelle</h3>
              <div class="form-group">
                <label class="form-label">Montant à payer</label>
                <input type="text" class="form-input" value="50 000 FCFA" disabled />
              </div>
              <div class="form-group">
                <label class="form-label">Mode de Paiement</label>
                <select class="form-select" id="pay-method">
                  <option value="Wave">Wave Mobile Money</option>
                  <option value="Orange Money">Orange Money</option>
                  <option value="Carte Bancaire">Carte Bancaire Visa / Mastercard</option>
                </select>
              </div>
              <div class="form-group">
                <label class="form-label">Numéro de Téléphone / Débit</label>
                <input type="tel" class="form-input" placeholder="+221 77 XXX XX XX" id="pay-phone" value="+221 77 123 45 67" />
              </div>
              <button class="btn-primary" style="width:100%; justify-content:center; padding:14px; background:linear-gradient(135deg,#22C55E,#16A34A);" onclick="processPayment()">
                Confirmer le Paiement
              </button>
            </div>

            <div>
              <h3 style="font-family:var(--font-heading); font-size:18px; font-weight:700; margin-bottom:20px; color:#0F172A;">Historique des Quittances</h3>
              <div class="table-container">
                <table class="data-table">
                  <thead>
                    <tr>
                      <th>Reçu</th>
                      <th>Mois</th>
                      <th>Montant</th>
                      <th>Action</th>
                    </tr>
                  </thead>
                  <tbody id="payment-receipts-body">
                    <tr>
                      <td>#REC-041</td>
                      <td>Juillet 2026</td>
                      <td>50 000 FCFA</td>
                      <td><a href="javascript:void(0)" style="color:#2563EB; font-size:12px; font-weight:600;" onclick="downloadReceipt()">Télécharger PDF</a></td>
                    </tr>
                  </tbody>
                </table>
              </div>
            </div>
          </div>
        </div>

        <!-- TAB 6: SIGNALEMENT INCIDENTS -->
        <div class="tab-content" id="tab-incidents">
          <div style="background:white; padding:28px; border-radius:20px; border:1px solid #E2E8F0; max-width:680px;">
            <h3 style="font-family:var(--font-heading); font-size:18px; font-weight:700; margin-bottom:20px; color:#0F172A;">Signaler un Incident Technique dans votre Local</h3>
            <form onsubmit="handleIncidentSubmit(event)">
              <div class="form-group">
                <label class="form-label">Type d'incident</label>
                <select class="form-select" id="inc-type" required>
                  <option value="Électricité">Électricité & Éclairage</option>
                  <option value="Plomberie">Plomberie & Fuite d'eau</option>
                  <option value="Serrure">Serrurerie & Porte</option>
                  <option value="Structure">Toiture / Structure du Kiosque</option>
                </select>
              </div>
              <div class="form-group">
                <label class="form-label">Description précise du problème</label>
                <textarea class="form-textarea" id="inc-desc" rows="4" placeholder="Décrivez la panne ou la réparation nécessaire..." required></textarea>
              </div>
              <button type="submit" class="btn-primary" style="background:#EF4444;">Transmettre à la Régie Technique</button>
            </form>
          </div>
        </div>

        <!-- TAB 7: ESPACE ADMIN CROUS THIES -->
        <div class="tab-content" id="tab-admin">
          <div style="display:grid; grid-template-columns: repeat(4,1fr); gap:16px; margin-bottom:24px;">
            <div style="background:white; padding:20px; border-radius:16px; border:1px solid #E2E8F0;">
              <div style="font-size:12px; color:#64748B;">Total Dossiers</div>
              <div style="font-size:26px; font-weight:800; color:#2563EB;">324</div>
            </div>
            <div style="background:white; padding:20px; border-radius:16px; border:1px solid #E2E8F0;">
              <div style="font-size:12px; color:#64748B;">En Attente Validation</div>
              <div style="font-size:26px; font-weight:800; color:#F59E0B;">12</div>
            </div>
            <div style="background:white; padding:20px; border-radius:16px; border:1px solid #E2E8F0;">
              <div style="font-size:12px; color:#64748B;">Recettes du Mois</div>
              <div style="font-size:26px; font-weight:800; color:#22C55E;">4.2M FCFA</div>
            </div>
            <div style="background:white; padding:20px; border-radius:16px; border:1px solid #E2E8F0;">
              <div style="font-size:12px; color:#64748B;">Taux d'Occupation</div>
              <div style="font-size:26px; font-weight:800; color:#7C3AED;">87%</div>
            </div>
          </div>

          <div class="table-container">
            <div style="padding:20px; border-bottom:1px solid #E2E8F0; font-weight:700; font-family:var(--font-heading);">Gestion Administrative des Demandes de Locaux</div>
            <table class="data-table">
              <thead>
                <tr>
                  <th>Demandeur</th>
                  <th>Projet / Commerce</th>
                  <th>Local Visé</th>
                  <th>Statut Dossier</th>
                  <th>Actions Admin</th>
                </tr>
              </thead>
              <tbody id="admin-table-body">
                <tr>
                  <td><strong>Moussa Diallo</strong><br><small style="color:#64748B;">+221 77 123 45 67</small></td>
                  <td>Papeterie / Impression</td>
                  <td>Local C-204</td>
                  <td><span class="status-badge pending" id="status-row-1">● En Attente</span></td>
                  <td>
                    <button style="padding:6px 12px; border-radius:8px; background:#22C55E; color:white; font-size:12px; font-weight:600;" onclick="adminApprove(1)">Approuver</button>
                    <button style="padding:6px 12px; border-radius:8px; background:#EF4444; color:white; font-size:12px; font-weight:600;" onclick="adminReject(1)">Rejeter</button>
                  </td>
                </tr>
                <tr>
                  <td><strong>Fatou Sall</strong><br><small style="color:#64748B;">+221 76 987 65 43</small></td>
                  <td>Kiosque Jus & Snacks</td>
                  <td>Kiosque B12</td>
                  <td><span class="status-badge approved">✓ Validé</span></td>
                  <td><span style="font-size:12px; color:#64748B;">Dossier Traité</span></td>
                </tr>
              </tbody>
            </table>
          </div>
        </div>

      </div>
    </main>
  </div>
</div>

<!-- TOAST NOTIFICATION CONTAINER -->
<div class="toast-container" id="toast-container"></div>

<!-- JAVASCRIPT LOGIC & INTERACTIVITY -->
<script>
  // 1. SCROLL & MOBILE MENU INTERACTION
  const nav = document.getElementById('navbar');
  window.addEventListener('scroll', () => {
    nav.classList.toggle('scrolled', window.scrollY > 20);
  });

  const ham = document.getElementById('hamburger');
  const mob = document.getElementById('mobile-menu');
  ham.addEventListener('click', () => mob.classList.toggle('open'));
  mob.querySelectorAll('a').forEach(a => a.addEventListener('click', () => mob.classList.remove('open')));

  // 2. FAQ ACCORDION TOGGLE
  document.querySelectorAll('.faq-q').forEach(btn => {
    btn.addEventListener('click', () => {
      const item = btn.closest('.faq-item');
      const isOpen = item.classList.contains('open');
      document.querySelectorAll('.faq-item').forEach(i => i.classList.remove('open'));
      if (!isOpen) item.classList.add('open');
    });
  });

  // 3. CAMPUS INTERACTIVE MAP PINS DATA & CLICK HANDLERS
  const pins = [
    {id:1, x:20, y:25, name:'Local A1 (Bât. Admin)', type:'available', cat:'Services & Papeterie', price:'60 000 FCFA / mois'},
    {id:2, x:45, y:20, name:'Café Campus Sciences', type:'occupied', cat:'Alimentation', price:'50 000 FCFA / mois'},
    {id:3, x:70, y:30, name:'Kiosque Cosmétiques', type:'available', cat:'Cosmétiques & Beauté', price:'40 000 FCFA / mois'},
    {id:4, x:30, y:55, name:'TechStore Informatique', type:'occupied', cat:'Électronique', price:'55 000 FCFA / mois'},
    {id:5, x:60, y:50, name:'Kiosque Maintenance B2', type:'maintenance', cat:'Services Techniques', price:'35 000 FCFA / mois'},
    {id:6, x:75, y:65, name:'Resto Etudiant Express', type:'occupied', cat:'Alimentation', price:'70 000 FCFA / mois'},
    {id:7, x:15, y:70, name:'Local R-4 Résidences', type:'available', cat:'Multiservices', price:'45 000 FCFA / mois'},
  ];
  const tc = {
    available: {color:'#22C55E', label:'Disponible', bg:'#D1FAE5'},
    occupied: {color:'#2563EB', label:'Occupé', bg:'#DBEAFE'},
    maintenance: {color:'#F59E0B', label:'Maintenance', bg:'#FEF3C7'},
  };

  const mapEl = document.getElementById('campus-map');
  let activePin = null;

  pins.forEach(p => {
    const el = document.createElement('div');
    el.className = 'map-pin';
    el.style.left = p.x + '%';
    el.style.top = p.y + '%';
    el.style.background = tc[p.type].color;
    el.innerHTML = '<div class="pin-in"></div>';

    el.addEventListener('click', () => {
      if (activePin) activePin.classList.remove('act');
      if (activePin === el) {
        activePin = null;
        document.getElementById('map-detail').classList.remove('vis');
        return;
      }
      el.classList.add('act');
      activePin = el;

      const c = tc[p.type];
      document.getElementById('d-name').textContent = p.name;
      const tag = document.getElementById('d-tag');
      tag.textContent = c.label;
      tag.style.background = c.bg;
      tag.style.color = c.color;

      document.getElementById('d-cat').textContent = 'Catégorie : ' + p.cat;
      document.getElementById('d-price').textContent = p.price;
      document.getElementById('d-btn').style.display = (p.type === 'available') ? 'block' : 'none';
      document.getElementById('map-detail').classList.add('vis');
    });

    mapEl.appendChild(el);
  });

  // 4. DASHBOARD DRAWER & TAB NAVIGATION
  const drawer = document.getElementById('dashboard-drawer');

  function openDashboardPortal(tab = 'dashboard') {
    drawer.classList.add('active');
    switchTab(tab);
  }

  function closeDashboardPortal() {
    drawer.classList.remove('active');
  }

  const tabTitles = {
    'dashboard': 'Tableau de Bord Global',
    'demandes': 'Formulaire & Suivi des Demandes',
    'locaux': 'Catalogue des Locaux Commerciales',
    'contrats': 'Signature Électronique du Bail',
    'paiements': 'Paiement Loyer (Wave / Orange Money)',
    'incidents': 'Signalement de Panne ou Maintenance',
    'admin': 'Administration CROUS Thiès'
  };

  function switchTab(tabId) {
    document.querySelectorAll('.sidebar-item').forEach(item => {
      item.classList.remove('active');
    });
    const clickedItem = Array.from(document.querySelectorAll('.sidebar-item')).find(el => el.getAttribute('onclick')?.includes(tabId));
    if (clickedItem) clickedItem.classList.add('active');

    document.querySelectorAll('.tab-content').forEach(tc => tc.classList.remove('active'));
    const targetTab = document.getElementById('tab-' + tabId);
    if (targetTab) targetTab.classList.add('active');

    document.getElementById('tab-title').textContent = tabTitles[tabId] || 'Espace Gestion';
  }

  // 5. TOAST NOTIFICATIONS
  function showToast(message, type = 'info') {
    const container = document.getElementById('toast-container');
    const toast = document.createElement('div');
    toast.className = 'toast';
    toast.innerHTML = `<span>ℹ️</span> <div>${message}</div>`;
    container.appendChild(toast);
    setTimeout(() => {
      toast.style.opacity = '0';
      setTimeout(() => toast.remove(), 300);
    }, 4000);
  }

  // 6. FORM HANDLERS & ACTIONS
  function handleNewRequest(e) {
    e.preventDefault();
    const name = document.getElementById('req-name').value;
    const cat = document.getElementById('req-category').value;
    const loc = document.getElementById('req-location').value;

    const tbody = document.getElementById('user-requests-list');
    const newRow = document.createElement('tr');
    const ref = '#REQ-' + Math.floor(100 + Math.random() * 900);
    newRow.innerHTML = `
      <td><strong>${ref}</strong></td>
      <td>${cat}</td>
      <td>${loc}</td>
      <td><span class="status-badge pending">● En Examen</span></td>
    `;
    tbody.prepend(newRow);

    const dashBody = document.getElementById('dash-activities-body');
    const dashRow = document.createElement('tr');
    dashRow.innerHTML = `
      <td><strong>${ref}</strong></td>
      <td>${cat}</td>
      <td>${loc}</td>
      <td>${new Date().toLocaleDateString('fr-FR')}</td>
      <td><span class="status-badge pending">● En cours d'examen</span></td>
    `;
    dashBody.prepend(dashRow);

    showToast(`Demande ${ref} transmise au CROUS avec succès !`, 'success');
    document.getElementById('form-new-request').reset();
  }

  function signContract() {
    const statusText = document.getElementById('contract-status-text');
    const btn = document.getElementById('btn-sign-contract');
    statusText.textContent = '✓ Contrat Signé Électroniquement';
    statusText.style.color = '#22C55E';
    btn.textContent = '✓ Signature Effectuée';
    btn.disabled = true;
    btn.style.background = '#94A3B8';
    showToast('Le contrat de bail a été signé avec succès !', 'success');
  }

  function processPayment() {
    const method = document.getElementById('pay-method').value;
    const phone = document.getElementById('pay-phone').value;

    showToast(`Paiement de 50 000 FCFA validé via ${method} (${phone})`, 'success');

    const receiptsBody = document.getElementById('payment-receipts-body');
    const newReceipt = document.createElement('tr');
    const ref = '#REC-' + Math.floor(100 + Math.random() * 900);
    newReceipt.innerHTML = `
      <td><strong>${ref}</strong></td>
      <td>Août 2026</td>
      <td>50 000 FCFA</td>
      <td><a href="javascript:void(0)" style="color:#2563EB; font-size:12px; font-weight:600;" onclick="downloadReceipt()">Télécharger PDF</a></td>
    `;
    receiptsBody.prepend(newReceipt);
  }

  function downloadReceipt() {
    showToast('Téléchargement du reçu de quittance en cours...', 'info');
  }

  function handleIncidentSubmit(e) {
    e.preventDefault();
    const type = document.getElementById('inc-type').value;
    showToast(`Ticket de signalement #${Math.floor(1000 + Math.random() * 9000)} transmis pour : ${type}`, 'warning');
    e.target.reset();
  }

  function adminApprove(id) {
    const badge = document.getElementById(`status-row-${id}`);
    if (badge) {
      badge.className = 'status-badge approved';
      badge.textContent = '✓ Approuvé';
    }
    showToast('Dossier approuvé et notifié à l\'usager !', 'success');
  }

  function adminReject(id) {
    const badge = document.getElementById(`status-row-${id}`);
    if (badge) {
      badge.className = 'status-badge rejected';
      badge.textContent = '✕ Rejeté';
    }
    showToast('Dossier marqué comme rejeté.', 'danger');
  }

  let isAdmin = false;
  function toggleUserRole() {
    isAdmin = !isAdmin;
    if (isAdmin) {
      document.getElementById('portal-avatar').textContent = 'KS';
      document.getElementById('portal-user-name').textContent = 'Dr. Khady Sall';
      document.getElementById('portal-user-role').textContent = 'Administratrice CROUS';
      switchTab('admin');
      showToast('Mode Administrateur CROUS activé', 'info');
    } else {
      document.getElementById('portal-avatar').textContent = 'MD';
      document.getElementById('portal-user-name').textContent = 'Moussa Diallo';
      document.getElementById('portal-user-role').textContent = 'Usager / Commerçant';
      switchTab('dashboard');
      showToast('Mode Usager activé', 'info');
    }
  }
</script>
</body>
</html>
""")



Pour utiliser `ngrok`, vous devez d'abord l'installer et fournir un jeton d'authentification. Si vous n'avez pas de compte `ngrok` ou de jeton, veuillez visiter leur site web pour vous inscrire et obtenir votre jeton d'authentification.

Vous pouvez l'ajouter aux secrets Colab (icône "🔑" dans le panneau de gauche) sous le nom `NGROK_AUTH_TOKEN` ou le coller directement dans la cellule ci-dessous.


In [ ]:
# 1. Lancez un serveur HTTP léger en arrière-plan.
# Il est crucial que ce serveur reste en cours d'exécution pour que ngrok puisse s'y connecter.
# Nous redirigeons la sortie vers un fichier log pour éviter de bloquer la cellule.
# Assurez-vous que le fichier index.html est bien créé par la cellule précédente (uw7oRIBaWiMx).
!python3 -m http.server 8000 --directory /content > /tmp/server.log 2>&1 &

print("Serveur HTTP démarré sur le port 8000. Vous pouvez maintenant lancer la cellule ngrok.")

In [ ]:
import subprocess

result = subprocess.run(
    ["npm", "install", "-g", "localtunnel"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)


changed 22 packages in 3s

3 packages are looking for funding
  run `npm fund` for details




In [ ]:
import subprocess

tunnel = subprocess.Popen(
    ["lt", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in tunnel.stdout:
    print(line, end="")

your url is: https://fresh-hounds-flash.loca.lt


In [ ]:
import os

print(os.path.exists("/content/index.html"))
print(os.path.getsize("/content/index.html"), "octets")